# Step 07 — XAI DGT intermediate 64 dan 128

**Utama: jam 48. Pembanding: jam 24. Split: 70:15:15.** Dua checkpoint empat lapisan,
LR 0,0001, k10; bobot tidak dilatih ulang. Epoch terbaik dibaca dari catatan training.

Simpan di `icu_los/notebooks`, pilih kernel CUDA training, lalu jalankan **Run All**.
Awali dengan `MODE="pilot"` (satu kasus per kelompok). Setelah meninjau hasil dan status
`PILOT_COMPLETE`, ubah hanya `MODE="full"` untuk memproses seluruh sampel. Penjelasan
pilot yang sudah selesai akan digunakan kembali. Pilot bukan hasil global akhir.

`DGT128_RUN` diisi dengan run dimensi 128 yang sudah kamu gunakan. `BASELINE_RUN`
adalah run dasar dimensi 64. Jika lokasi folder proyek berbeda, sesuaikan `ROOT`.

Hasil disimpan pada `outputs/07_xai_70_15_15/<identitas>`; jika dihentikan, Run All
melanjutkan unit kasus/model/landmark yang belum selesai. Satu unit yang terputus
akan dihitung dari awal. Perubahan pengaturan numerik atau sampel membuat laporan baru.

## Makna dan batas analisis

- IG menjelaskan **fitur pasien utama**; masukan tetangga, mask, dan jam tetap aktual.
- Seluruh prefix dihitung ulang dari jam 1, dengan gradient melewati state antarblok.
- Graf tetap memakai snapshot aktual yang berubah per jam. Graf tidak dibangun ulang
  sepanjang lintasan IG; analisis hubungan dipisahkan melalui edge deletion.
- Baseline numerik z0, kategori modal train, embedding teks0. Baseline bukan pasien
  klinis normal. Interpolasi kategori hanya lintasan matematis pada representasi onehot.
- Atribusi dalam hari terhadap prediksi **total LOS**, bukan bukti sebab akibat.
- Teks dijelaskan pada tingkat embedding; tidak ada sorotan token atau klaim makna
  klinis untuk dimensi embedding. Catatan mentah tidak diekspor.
- Peringkat fitur merangkum kontribusi bersih sepanjang waktu/dimensi terlebih dahulu,
  baru mengambil nilai absolut. Kontribusi berlawanan dapat saling meniadakan.
- Sampel berimbang LOS tidak mewakili prevalensi seluruh test set. Kelompok pendek
  hanya mencakup kasus yang masih memiliki prediksi valid pada jam 48.
- Kuota 30 per kelompok, enam kasus dokter, cut-off 3/7 hari, dan toleransi numerik
  merupakan konfigurasi penelitian yang dapat ditinjau, bukan angka baku tervalidasi.

**Komputasi:** satu titik integrasi pada satu waktu; activation checkpoint membatasi
memori tanpa memutus gradient. CUDA memakai backend LSTM yang mendukung backward
pada mode eval; prediksinya wajib lolos pemeriksaan terhadap evaluator acuan.
Notebook belum diuji pada GPU/data klinis milik pengguna.

In [39]:
from pathlib import Path

ROOT = Path(r"C:\Users\ASUS\Documents\icu_los")
BASELINE_RUN = "20261103_001451_290579_models_70_15_15"
DGT128_RUN = "20261103_001451_290579_models_70_15_15_dgt_inter_d128_cf317322"
MODE = "full"  # Setelah PILOT_COMPLETE, ganti menjadi "full".

XAI_SETTINGS = dict(
    seed=42,
    hours=[48, 24],
    los_cutoffs_days=[3, 7],
    per_group=30,
    doctor_per_group=2,
    ig_steps=32,
    ig_max_steps=128,
    ig_abs_tolerance_days=0.001,
    ig_rel_tolerance=0.01,
    ig_stability_tolerance=0.10,
    prediction_atol=0.0001,
    prediction_rtol=0.00001,
    checkpoint_blocks=True,
    fidelity_counts=[1, 3, 5],
    random_repeats=5,
    graph_neighbours_to_probe=5,
)

## Definisi model Step 05

Salinan definisi acuan diperlukan untuk memeriksa fingerprint serta memuat model dan
input yang sama dengan training. Sel training tidak dipanggil oleh notebook ini.

In [40]:
import os

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from datetime import datetime, timezone
from contextlib import contextmanager
from functools import lru_cache
import copy
import gc
import hashlib
import inspect
import json
import math
import platform
import random
import shutil
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display

FAMILIES = ["LSTM", "StaticGraphTransformer", "DGT"]
FUSIONS = ["early", "intermediate", "late"]
ROLES = {"train": 0, "validation": 1, "test": 2}
LANDMARKS = (4, 24, 48)
GRAPH_K = 10
GRAPH_VERIFIED = {}
MODEL_TEXT_ARRAYS = [
    "icustay_ids_LOCAL.npy",
    "note_interval_vectors_LOCAL.npy",
    "note_interval_index_LOCAL.npy",
    "note_stay_offsets_LOCAL.npy",
    "diagnosis_vectors_LOCAL.npy",
    "diagnosis_present_LOCAL.npy",
    "diagnosis_available_hour_LOCAL.npy",
]

In [41]:
LSTM_CONFIG = dict(
    name="baseline_lstm",
    d_model=64,
    dropout=0.1,
    lr=0.001,
    weight_decay=0.0,
    loss="mse",
    heads=None,
    graph_layers=0,
    k=0,
)

In [42]:
SGT_CONFIG = dict(
    name="baseline_sgt",
    d_model=64,
    dropout=0.4,
    lr=0.0001,
    weight_decay=0.0,
    loss="mse",
    heads=8,
    graph_layers=4,
    k=10,
)

In [43]:
DGT_CONFIGS = [
    dict(
        name="lr_001",
        d_model=64,
        dropout=0.2,
        lr=0.001,
        weight_decay=0.00001,
        loss="mse",
        heads=8,
        graph_layers=4,
        k=10,
    ),
    dict(
        name="lr_0001",
        d_model=64,
        dropout=0.2,
        lr=0.0001,
        weight_decay=0.00001,
        loss="mse",
        heads=8,
        graph_layers=4,
        k=10,
    ),
]
MODEL_CONFIGS = {
    "LSTM": [LSTM_CONFIG],
    "StaticGraphTransformer": [SGT_CONFIG],
    "DGT": DGT_CONFIGS,
}

In [44]:
def require(ok, message):
    if not bool(ok):
        raise ValueError(message)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))


def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(tmp, path)


def sha(path):
    h = hashlib.sha256()
    last = time.monotonic()
    done = 0
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(b)
            done += len(b)
            if time.monotonic() - last > 25:
                print(f"Hash scan {done / 2 ** 30:.2f} GiB", flush=True)
                last = time.monotonic()
    return h.hexdigest()


def artifact(p):
    p = Path(p)
    st = p.stat()
    return dict(
        path=str(p.resolve()), bytes=st.st_size, mtime_ns=st.st_mtime_ns, sha256=sha(p)
    )


def atomic_torch(path, obj):
    p = Path(path)
    tmp = p.with_suffix(p.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, p)


def atomic_npy(path, a):
    p = Path(path)
    tmp = p.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        np.save(f, a, allow_pickle=False)
    os.replace(tmp, p)


@contextmanager
def run_lock(path):
    f = open(path, "a+b")
    f.seek(0)
    f.write(b"0")
    f.flush()
    f.seek(0)
    try:
        if os.name == "nt":
            import msvcrt

            msvcrt.locking(f.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl

            fcntl.flock(f.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
    except OSError:
        f.close()
        raise RuntimeError("Run aktif pada kernel lain; hentikan proses lain dahulu")
    try:
        yield
    finally:
        f.seek(0)
        if os.name == "nt":
            msvcrt.locking(f.fileno(), msvcrt.LK_UNLCK, 1)
        else:
            fcntl.flock(f.fileno(), fcntl.LOCK_UN)
        f.close()


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def rng_state():
    return dict(
        python=random.getstate(),
        numpy=np.random.get_state(),
        torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    )


def restore_rng(r):
    random.setstate(r["python"])
    np.random.set_state(r["numpy"])
    torch.set_rng_state(r["torch"])
    if r["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(r["cuda"])


def tree_map(x, fn):
    if isinstance(x, torch.Tensor):
        return fn(x)
    if isinstance(x, dict):
        return {k: tree_map(v, fn) for k, v in x.items()}
    if isinstance(x, (tuple, list)):
        return type(x)((tree_map(v, fn) for v in x))
    return x


def digest(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, allow_nan=False).encode()
    ).hexdigest()


def runtime_code_hash():
    """Hash fungsi yang aktif, termasuk metode class; tidak bergantung nama file notebook."""
    import types

    def encode(value):
        if isinstance(value, types.CodeType):
            return dict(
                code=value.co_code.hex(),
                constants=[encode(v) for v in value.co_consts],
                names=value.co_names,
                variables=value.co_varnames,
                free=value.co_freevars,
                cells=value.co_cellvars,
                args=[value.co_argcount, value.co_posonlyargcount, value.co_kwonlyargcount],
                flags=value.co_flags,
                exceptions=value.co_exceptiontable.hex(),
            )
        if isinstance(value, bytes):
            return value.hex()
        if isinstance(value, (list, tuple)):
            return [encode(v) for v in value]
        if isinstance(value, frozenset):
            return sorted([encode(v) for v in value], key=repr)
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        return repr(value)

    payload = {}
    for name in ENGINE_NAMES:
        value = globals()[name]
        if isinstance(value, type):
            payload[name] = {
                key: encode(inspect.unwrap(method).__code__)
                for key, method in value.__dict__.items()
                if hasattr(inspect.unwrap(method), "__code__")
            }
        else:
            payload[name] = encode(inspect.unwrap(value).__code__)
    return digest(payload)

In [45]:
def verify_sources(root, stage03_run, stage04_run, data):
    """Periksa paket holdout, seluruh fitur yang dipakai, dan pembagian pasien."""
    r3, r4 = [Path(root) / "outputs" / run for run in [stage03_run, stage04_run]]
    m3, m4 = [read_json(folder / "run_metadata.json") for folder in [r3, r4]]
    require(
        m3["status"] == "SPLIT_PREPROCESS_COMPLETE_READY_FOR_TEXT_EMBEDDINGS",
        "Step03 belum selesai",
    )
    require(
        m4["status"] == "TEXT_EMBEDDINGS_COMPLETE_READY_FOR_MODEL_DESIGN",
        "Step04 belum selesai",
    )
    for folder in [r3, r4]:
        q = pd.read_csv(folder / "quality_checks.csv")
        require(
            len(q) > 0 and q.status.eq("PASS").all(), "Quality checks sumber belum PASS"
        )
    p3, p4 = (Path(m3["input_package"]), Path(m4["text_input_package"]))
    require(
        sha(p3) == m3["input_package_sha256"]
        and sha(p4) == m4["text_input_package_sha256"],
        "Hash paket berubah",
    )
    numeric, text = (read_json(p3), read_json(p4))
    require(
        numeric["version"] == "icu_los_v2_stage03_holdout_70_15_15"
        and text["version"] == "icu_los_v2_stage04_holdout_70_15_15",
        "Gunakan paket holdout baru",
    )
    require(
        numeric["fit_scope"] == "TRAIN_ONLY_HOLDOUT"
        and (not numeric["target_feature"])
        and (not text["train_fitted_transform"])
        and (not text["LOS_target_in_X"]),
        "Kontrak train-only berbeda",
    )
    require(
        numeric["cohort_sha256"]
        == text["cohort_sha256"]
        == m3["cohort_sha256"]
        == m4["cohort_sha256"],
        "Cohort berbeda",
    )
    require(
        text["stage03_input_package_sha256"] == m3["input_package_sha256"],
        "Embedding belum terhubung ke Step03 ini",
    )
    require(
        numeric["split_ratios"]
        == text["split_ratios"]
        == {"train": 0.7, "validation": 0.15, "test": 0.15},
        "Rasio split berbeda",
    )
    receipt = read_json(r4 / "reuse_receipt.json")
    entries = (
        m3["output_artifacts"]
        + m3["sources"]
        + m4["artifacts"]
        + receipt["source_artifacts"]
    )
    expected = {str(Path(a["path"]).resolve()): a for a in entries}
    paths = [
        r3 / "run_metadata.json",
        r4 / "run_metadata.json",
        p3,
        p4,
        Path(numeric["row_map"]),
        Path(numeric["sequence_index"]),
        Path(numeric["split_assignments"]),
        Path(numeric["y"]),
        Path(numeric["normalized_partition_ledger"]),
    ]
    paths += [
        Path(numeric["structured_inputs"][key])
        for key in ["static_file", "parameters_file"]
    ]
    ledger = pd.read_csv(numeric["normalized_partition_ledger"])
    require(len(ledger) > 0 and ledger.partition.is_unique, "Ledger partisi tidak valid")
    for row in ledger.itertuples():
        paths += [Path(row.raw_path), Path(row.normalized_path)]
    paths += [Path(text["representations"][name]) for name in MODEL_TEXT_ARRAYS]
    paths += [
        r3 / "quality_checks.csv",
        r4 / "quality_checks.csv",
        r4 / "reuse_receipt.json",
    ]
    old_path = Path(data) / "source_verification.json"
    previous = read_json(old_path) if old_path.exists() else {}
    watch = {}
    for i, path in enumerate(dict.fromkeys(paths)):
        key = str(path.resolve())
        stat = path.stat()
        entry = expected.get(key)
        require(
            entry is not None or path.name == "run_metadata.json",
            "Sumber tidak tercatat: " + path.name,
        )
        prior = previous.get(key)
        if (
            prior
            and prior["bytes"] == stat.st_size
            and (prior["mtime_ns"] == stat.st_mtime_ns)
        ):
            checksum = prior["sha256"]
        else:
            checksum = sha(path)
        require(
            entry is None
            or (checksum == entry["sha256"] and stat.st_size == entry["bytes"]),
            "Sumber berubah: " + str(path),
        )
        watch[key] = dict(bytes=stat.st_size, mtime_ns=stat.st_mtime_ns, sha256=checksum)
        if i % 50 == 0:
            print(f"Input integrity {i + 1}/{len(set(paths))} files", flush=True)
    sample, sequence, splits = [
        pd.read_csv(numeric[key])
        for key in ["row_map", "sequence_index", "split_assignments"]
    ]
    require(
        sample.ICUSTAY_ID.is_unique
        and np.array_equal(sample.cohort_row, np.arange(len(sample))),
        "Row map invalid",
    )
    for frame in [sequence, splits]:
        require(
            len(frame) == len(sample)
            and np.array_equal(frame.cohort_row, sample.cohort_row)
            and np.array_equal(frame.ICUSTAY_ID, sample.ICUSTAY_ID),
            "Urutan input berbeda",
        )
    require(
        np.array_equal(splits.SUBJECT_ID, sample.SUBJECT_ID)
        and splits.groupby("SUBJECT_ID").split.nunique().eq(1).all()
        and splits.groupby("HADM_ID").split.nunique().eq(1).all()
        and (set(splits.split) == set(ROLES)),
        "Isolasi pasien/admission gagal",
    )
    require(
        sequence.first_prediction_hour.eq(4).all()
        and sequence.last_prediction_hour.ge(4).all(),
        "Jadwal prediksi invalid",
    )
    require(
        len(sample) == text["cohort_stays"] == m3["counts"]["cohort_stays"],
        "Jumlah cohort berbeda",
    )
    atomic_json(old_path, watch)
    return dict(
        numeric=numeric,
        text=text,
        sample=sample,
        seq=sequence,
        splits=splits,
        ledger=ledger,
        watch=watch,
        cohort_sha256=numeric["cohort_sha256"],
    )

In [46]:
class HoldoutData:
    """Input holdout: static, temporal, mask dan teks selaras menurut cohort_row; ID/target bukan fitur."""

    def __init__(self, inputs):
        self.inputs = inputs
        self.split_name = "holdout_70_15_15"
        self.sample = inputs["sample"]
        self.n = len(self.sample)
        self.seq = inputs["seq"]
        self.lengths = self.seq.last_prediction_hour.to_numpy(dtype=np.int64)
        self.subjects = self.sample.SUBJECT_ID.to_numpy(dtype=np.int64)
        self.role = (
            inputs["splits"]
            .sort_values("cohort_row")
            .split.map(ROLES)
            .to_numpy(dtype=np.int64)
        )
        self.y = np.load(inputs["numeric"]["y"], mmap_mode="r", allow_pickle=False)
        require(
            self.y.shape == (self.n,)
            and self.y.dtype == np.float64
            and np.isfinite(self.y).all()
            and (self.y > 4 / 24).all(),
            "Target invalid",
        )
        self.ledger = inputs["ledger"].sort_values("partition").set_index("partition")
        structured = inputs["numeric"]["structured_inputs"]
        with np.load(structured["static_file"], allow_pickle=False) as z:
            self.static = np.concatenate(
                [z["X_static"], z["mask_observed"]], axis=1
            ).astype(np.float32)
            self.feature_names = z["feature_names"].tolist() + [
                "observed:" + x for x in inputs["numeric"]["static_mask_features"]
            ]
        require(
            self.static.shape[0] == self.n and np.isfinite(self.static).all(),
            "Static input invalid",
        )
        self.temporal_params = read_json(structured["parameters_file"])["temporal"]
        self.S = self.static.shape[1]
        paths = inputs["text"]["representations"]
        self.text = {
            name: np.load(paths[name], mmap_mode="r", allow_pickle=False)
            for name in MODEL_TEXT_ARRAYS
        }
        require(
            np.array_equal(self.text["icustay_ids_LOCAL.npy"], self.sample.ICUSTAY_ID),
            "Urutan teks berbeda",
        )
        require(
            self.text["diagnosis_vectors_LOCAL.npy"].shape == (self.n, 768),
            "Shape diagnosis invalid",
        )
        index, offsets = (
            self.text["note_interval_index_LOCAL.npy"],
            self.text["note_stay_offsets_LOCAL.npy"],
        )
        require(
            index.ndim == 2
            and index.shape[1] == 3
            and (offsets.shape == (self.n + 1,))
            and (offsets[0] == 0)
            and (offsets[-1] == len(index)),
            "Offset teks invalid",
        )
        require(
            self.text["note_interval_vectors_LOCAL.npy"].shape == (len(index), 768),
            "Shape notes invalid",
        )

    @lru_cache(maxsize=512)
    def partition(self, p):
        r = self.ledger.loc[int(p)]
        with np.load(r.raw_path, allow_pickle=False) as z:
            row = z["cohort_row"].copy()
            offset = z["offsets"].copy()
            mask = z["mask_observed"].copy()
        values = np.load(r.normalized_path, mmap_mode="r", allow_pickle=False)
        require(
            values.shape == mask.shape
            and values.shape[1] == 43
            and (values.dtype == np.float32),
            "Partisi invalid",
        )
        require(
            np.array_equal(row, self.seq[self.seq.partition.eq(int(p))].cohort_row),
            "Urutan partisi berbeda",
        )
        return (row, offset, mask, values)

    def block(self, rows, start, stop):
        rows = np.asarray(rows, dtype=np.int64)
        N = len(rows)
        L = stop - start
        hours = np.arange(start + 1, stop + 1, dtype=np.float32)
        x = np.zeros((N, L, 43), np.float32)
        mask = np.zeros_like(x)
        notes = np.zeros((N, L, 768), np.float32)
        nm = np.zeros((N, L), np.float32)
        counts = np.zeros((N, L), np.int32)
        ix = self.text["note_interval_index_LOCAL.npy"]
        off = self.text["note_stay_offsets_LOCAL.npy"]
        for j, row in enumerate(rows):
            r = self.seq.iloc[row]
            _, offset, obs, values = self.partition(int(r.partition))
            local = int(r.local_stay_row)
            a = int(offset[local])
            end = int(offset[local + 1])
            length = end - a
            for q in range(43):
                if q < 30:
                    inds = np.minimum(np.arange(start, stop), length - 1)
                    x[j, :, q] = values[a + inds, q]
                else:
                    m = min(stop, length) - start
                    if m > 0:
                        x[j, :m, q] = values[a + start : a + start + m, q]
                    if m < L:
                        t = self.temporal_params
                        x[j, max(0, m) :, q] = (t["median"][q] - t["mean"][q]) / t["scale"][
                            q
                        ]
            m = min(stop, length) - start
            if m > 0:
                mask[j, :m] = obs[a + start : a + start + m]
            lo, hi = map(int, off[row : row + 2])
            group = ix[lo:hi]
            sel = (group[:, 1] >= start) & (group[:, 1] < stop)
            pos = group[sel, 1].astype(np.int64) - start
            if len(pos):
                ids = np.flatnonzero(sel) + lo
                notes[j, pos] = self.text["note_interval_vectors_LOCAL.npy"][ids]
                nm[j, pos] = 1
                counts[j, pos] = group[sel, 2]
        dp = np.asarray(self.text["diagnosis_present_LOCAL.npy"][rows], dtype=np.float32)
        dh = np.asarray(
            self.text["diagnosis_available_hour_LOCAL.npy"][rows], dtype=np.float64
        )
        dm = (
            (hours[None, :].astype(np.float64) > dh[:, None]) & dp[:, None].astype(bool)
        ).astype(np.float32)
        diag = np.asarray(
            self.text["diagnosis_vectors_LOCAL.npy"][rows], dtype=np.float32
        ).copy()
        batch = dict(
            static=self.static[rows].copy(),
            numeric=x,
            numeric_mask=mask,
            notes=notes,
            notes_mask=nm,
            diagnosis=diag,
            diagnosis_mask=dm,
            hours=hours,
        )
        require(
            all((np.isfinite(a).all() for a in batch.values())), "Model input tidak finite"
        )
        require(
            set(batch)
            == {
                "static",
                "numeric",
                "numeric_mask",
                "notes",
                "notes_mask",
                "diagnosis",
                "diagnosis_mask",
                "hours",
            },
            "Input whitelist gagal",
        )
        return (batch, counts)

    def close(self):
        self.partition.cache_clear()
        for value in list(self.text.values()) + [self.y]:
            if getattr(value, "_mmap", None) is not None:
                value._mmap.close()

In [47]:
def fixed_groups(n, size, seed):
    """Partition all stays into seeded, disjoint graph groups. Grouping never uses LOS, final sequence length or labels. The neighbour candidate pool is this group, not the whole cohort."""
    require(size >= 2, "Subgraph minimal dua node")
    order = np.random.default_rng(seed).permutation(n)
    return [order[i : i + size] for i in range(0, n, size)]


def normalize(x):
    return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), 1e-12)


def topk_graph(numeric, text, text_available, subjects, k, view):
    """Build directed neighbour → target edges by cosine similarity. Exclude self and other stays of the same patient; self attention is added later. Numeric/text scopes stay isolated for late fusion."""
    N = len(numeric)
    num = normalize(numeric)
    txt = normalize(text)
    valid = text_available[:, None] & text_available[None, :]
    scores = num @ num.T
    if view == "text":
        scores = txt @ txt.T
        scores = np.where(valid, scores, -np.inf)
    elif view == "balanced":
        scores = (scores + np.where(valid, txt @ txt.T, 0)) / (1 + valid)
    elif view != "numeric":
        raise ValueError("Graph view invalid")
    scores[subjects[:, None] == subjects[None, :]] = -np.inf
    order = np.argsort(-scores, axis=1, kind="stable")[:, : min(k, N)]
    e = np.full((N, k), -1, dtype=np.int32)
    w = np.zeros((N, k), np.float32)
    vals = np.take_along_axis(scores, order, axis=1)
    good = np.isfinite(vals)
    e[:, : order.shape[1]] = np.where(good, order, -1)
    w[:, : order.shape[1]] = np.where(good, np.exp(np.where(good, vals, 0)), 0)
    return (e, w)

In [48]:
def graph_cache(ds, rows, folder, kmax, chunk=16, until=None):
    """Build causal hourly graph snapshots once per holdout/group. Topology uses structured features plus cumulative available note context. Train/validation/test FEATURES can be neighbours in this transductive design; labels do not build edges."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    commit = folder / "commit.json"
    total = int(ds.lengths[rows].max())
    total = min(total, until) if until else total
    N = len(rows)
    require(N < 32768, "Subgraph terlalu besar untuk indeks int16")
    identity = dict(
        rows=np.asarray(rows).tolist(),
        split_name=ds.split_name,
        total_hours=total,
        kmax=kmax,
        chunk=chunk,
        seed=1729,
        protocol="fixed_subgraph_transductive_relative_hour_no_outcome_topology",
    )
    if commit.exists():
        c = read_json(commit)
        require(c["identity"] == identity, "Kontrak graf cache berbeda")
        for a in c["files"]:
            st = Path(a["path"]).stat()
            key = (a["path"], st.st_size, st.st_mtime_ns, a["sha256"])
            if key not in GRAPH_VERIFIED:
                require(
                    st.st_size == a["bytes"] and sha(a["path"]) == a["sha256"],
                    "Graf cache berubah",
                )
                GRAPH_VERIFIED[key] = True
        return {
            v: (
                np.load(folder / (v + "_edges.npy"), mmap_mode="r"),
                np.load(folder / (v + "_weights.npy"), mmap_mode="r"),
            )
            for v in ["balanced", "numeric", "text"]
        }
    require(shutil.disk_usage(folder).free / 2**30 > 2, "Disk kurang saat membentuk graf")
    edges = {
        v: np.lib.format.open_memmap(
            folder / (v + "_edges.npy"), mode="w+", dtype="int16", shape=(total, N, kmax)
        )
        for v in ["balanced", "numeric", "text"]
    }
    weights = {
        v: np.lib.format.open_memmap(
            folder / (v + "_weights.npy"),
            mode="w+",
            dtype="float32",
            shape=(total, N, kmax),
        )
        for v in edges
    }
    projection = np.random.default_rng(1729).normal(size=(768, 32)).astype(
        np.float32
    ) / np.sqrt(32)
    sums = np.zeros((N, 32), np.float64)
    counts = np.zeros(N, np.float64)
    lastlog = time.monotonic()
    for start in range(0, total, chunk):
        stop = min(total, start + chunk)
        b, nc = ds.block(rows, start, stop)
        pn = b["notes"] @ projection
        pdg = b["diagnosis"] @ projection
        for j, h in enumerate(range(start, stop)):
            sums += pn[:, j] * nc[:, j, None]
            counts += nc[:, j]
            dm = b["diagnosis_mask"][:, j]
            tc = (sums + pdg * dm[:, None]) / np.maximum(counts + dm, 1)[:, None]
            numeric = np.concatenate(
                [b["static"], b["numeric"][:, j], b["numeric_mask"][:, j]], axis=1
            )
            for view in edges:
                e, w = topk_graph(
                    numeric, tc, counts + dm > 0, ds.subjects[rows], kmax, view
                )
                edges[view][h] = e
                weights[view][h] = w
        if time.monotonic() - lastlog > 25:
            print(f"Graph cache {folder.name}: {stop:,}/{total:,} jam", flush=True)
            lastlog = time.monotonic()
    files = []
    for d in [edges, weights]:
        for a in d.values():
            a.flush()
    edges.clear()
    weights.clear()
    for p in sorted(folder.glob("*.npy")):
        files.append(artifact(p))
    atomic_json(commit, dict(identity=identity, files=files))
    return {
        v: (
            np.load(folder / (v + "_edges.npy"), mmap_mode="r"),
            np.load(folder / (v + "_weights.npy"), mmap_mode="r"),
        )
        for v in ["balanced", "numeric", "text"]
    }


def graph_block(cache, family, start, stop, k, device):
    """DGT reads each hourly topology; Static Graph Transformer repeats topology from endpoint 4h. Before endpoint 4h, external edges are absent."""
    result = {}
    for v, (e, w) in cache.items():
        if family == "StaticGraphTransformer":
            ids = np.maximum(np.arange(start, stop), 3)
            ids[:] = 3
            ee = np.asarray(e[ids, :, :k]).copy()
            ww = np.asarray(w[ids, :, :k]).copy()
            for j, h in enumerate(range(start, stop)):
                if h < 3:
                    ee[j] = -1
                    ww[j] = 0
        else:
            ee = np.asarray(e[start:stop, :, :k]).copy()
            ww = np.asarray(w[start:stop, :, :k]).copy()
        result[v] = (
            torch.as_tensor(ee, dtype=torch.long, device=device),
            torch.as_tensor(ww, dtype=torch.float32, device=device),
        )
    return result


def tensor_batch(b, device):
    return {k: torch.as_tensor(v, dtype=torch.float32, device=device) for k, v in b.items()}

In [49]:
class SpatialAttention(nn.Module):
    """One sparse multi-head graph attention block. Query comes from target, keys/values from self and valid neighbours; similarity prior enters logits. Residual paths, LayerNorm and FFN refine the representation."""

    def __init__(self, d, heads, dropout):
        super().__init__()
        require(d % heads == 0, "heads harus membagi d_model")
        self.heads = heads
        self.d = d
        self.q = nn.Linear(d, d)
        self.k = nn.Linear(d, d)
        self.v = nn.Linear(d, d)
        self.out = nn.Linear(d, d)
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(
            nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * d, d)
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x, edge, weight):
        N, L, D = x.shape
        H = self.heads
        dh = D // H
        h = x.transpose(0, 1)
        e = edge.clamp_min(0)
        batch = torch.arange(L, device=x.device)[:, None, None]
        neighbours = h[batch, e]
        neighbours = torch.cat([h[:, :, None, :], neighbours], dim=2)
        valid = torch.cat(
            [torch.ones((L, N, 1), dtype=torch.bool, device=x.device), edge >= 0], dim=2
        )
        prior = torch.cat([torch.ones((L, N, 1), device=x.device), weight], dim=2)
        q = self.q(h).reshape(L, N, H, dh)
        k = self.k(neighbours).reshape(L, N, -1, H, dh)
        v = self.v(neighbours).reshape(L, N, -1, H, dh)
        score = (q[:, :, None] * k).sum(-1) / math.sqrt(dh) + prior.clamp_min(1e-12).log()[
            ..., None
        ]
        score = score.masked_fill(~valid[..., None], -torch.inf)
        alpha = score.softmax(dim=2)
        z = (self.drop(alpha)[..., None] * v).sum(2).reshape(L, N, D)
        a = self.norm1(h + self.drop(self.out(z)))
        a = self.norm2(a + self.drop(self.ff(a)))
        return a.transpose(0, 1)

In [50]:
class Backbone(nn.Module):
    """LSTM baseline: causal recurrent layer only. Graph models: four spatial blocks, then causal LSTM and one local temporal attention layer (past 32 states). Recurrent state carries older compressed history across blocks."""

    def __init__(self, family, d, heads, dropout, attention_window, graph_layers=4):
        super().__init__()
        self.family = family
        self.window = attention_window
        self.rnn = nn.LSTM(d, d, batch_first=True)
        self.output_dropout = nn.Dropout(dropout)
        if family != "LSTM":
            require(
                graph_layers == 4, "Graph Transformer harus empat lapisan pada protokol ini"
            )
            self.graph = nn.ModuleList(
                [SpatialAttention(d, heads, dropout) for _ in range(graph_layers)]
            )
            self.attn = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
            self.n1 = nn.LayerNorm(d)
            self.n2 = nn.LayerNorm(d)
            self.ff = nn.Sequential(
                nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * d, d)
            )
            self.drop = nn.Dropout(dropout)

    def forward(self, x, graph, state):
        state = state or {}
        z = x
        if self.family != "LSTM":
            for layer in self.graph:
                z = layer(z, *graph)
        z, h = self.rnn(z, state.get("h"))
        new = {"h": h}
        if self.family == "LSTM":
            return (self.output_dropout(z), new)
        memory = state.get("memory")
        joined = torch.cat([memory, z], 1) if memory is not None else z
        length = joined.shape[1]
        pos = torch.arange(length, device=x.device)
        mask = (pos[None, :] > pos[:, None]) | (pos[:, None] - pos[None, :] >= self.window)
        a, _ = self.attn(joined, joined, joined, attn_mask=mask, need_weights=False)
        a = self.n1(joined + self.drop(a))
        a = self.n2(a + self.drop(self.ff(a)))
        new["memory"] = (
            joined[:, -(self.window - 1) :] if self.window > 1 else joined[:, :0]
        )
        return (a[:, -z.shape[1] :], new)

In [51]:
class FusionEncoder(nn.Module):
    """Early: concatenate static, numeric, masks, available text and elapsed time then project jointly. Intermediate: encode numeric and text separately, then join. Late: return separate numeric/text representations."""

    def __init__(self, S, d, fusion, dropout):
        super().__init__()
        self.fusion = fusion
        if fusion == "early":
            self.joint = nn.Sequential(
                nn.Linear(S + 86 + 1538 + 1, d),
                nn.LayerNorm(d),
                nn.GELU(),
                nn.Dropout(dropout),
            )
        else:
            self.static = nn.Sequential(nn.Linear(S, d), nn.GELU(), nn.Dropout(dropout))
            self.numeric_rnn = nn.LSTM(87, d, batch_first=True)
            self.numeric_joint = nn.Sequential(
                nn.Linear(2 * d, d), nn.LayerNorm(d), nn.GELU()
            )
            self.diag = nn.Linear(768, d // 2)
            self.notes = nn.Linear(768, d // 2)
            self.text_rnn = nn.LSTM(d + 3, d, batch_first=True)
            if fusion == "intermediate":
                self.joint = nn.Sequential(
                    nn.Linear(2 * d, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(dropout)
                )

    def forward(self, b, state=None):
        state = state or {}
        N, L, _ = b["numeric"].shape
        static = b["static"][:, None, :].expand(-1, L, -1)
        elapsed = torch.log1p(b["hours"])[None, :, None].expand(N, -1, -1) / 10
        dm = b["diagnosis_mask"][..., None]
        nm = b["notes_mask"][..., None]
        diag = b["diagnosis"][:, None, :].expand(-1, L, -1) * dm
        notes = b["notes"] * nm
        if self.fusion == "early":
            return (
                self.joint(
                    torch.cat(
                        [
                            static,
                            b["numeric"],
                            b["numeric_mask"],
                            diag,
                            dm,
                            notes,
                            nm,
                            elapsed,
                        ],
                        -1,
                    )
                ),
                {},
            )
        num, hnum = self.numeric_rnn(
            torch.cat([b["numeric"], b["numeric_mask"], elapsed], -1), state.get("numeric")
        )
        num = self.numeric_joint(
            torch.cat([self.static(b["static"])[:, None, :].expand(-1, L, -1), num], -1)
        )
        txt, htxt = self.text_rnn(
            torch.cat([self.diag(diag) * dm, self.notes(notes) * nm, dm, nm, elapsed], -1),
            state.get("text"),
        )
        st = {"numeric": hnum, "text": htxt}
        return (
            (num, txt) if self.fusion == "late" else self.joint(torch.cat([num, txt], -1)),
            st,
        )

In [52]:
class LOSModel(nn.Module):
    """Predict TOTAL ICU LOS in days at every endpoint. Head = elapsed days + softplus(remaining days). Late fusion combines two predictions with a sigmoid-constrained scalar weight."""

    def __init__(self, S, family, fusion, cfg, attention_window=32):
        super().__init__()
        self.family = family
        self.fusion = fusion
        d = cfg["d_model"]
        self.encoder = FusionEncoder(S, d, fusion, cfg["dropout"])

        def backbone():
            return Backbone(
                family,
                d,
                cfg["heads"],
                cfg["dropout"],
                attention_window,
                cfg.get("graph_layers", 4),
            )

        self.num = backbone()
        self.head_num = nn.Linear(d, 1)
        if fusion == "late":
            self.txt = backbone()
            self.head_txt = nn.Linear(d, 1)
            self.mix = nn.Parameter(torch.zeros(()))

    def forward(self, b, graphs=None, state=None):
        state = state or {}
        h, enc = self.encoder(b, state.get("enc"))
        hours = b["hours"][None, :] / 24
        if self.fusion == "late":
            n, ns = self.num(
                h[0], graphs.get("numeric") if graphs else None, state.get("num")
            )
            t, ts = self.txt(h[1], graphs.get("text") if graphs else None, state.get("txt"))
            pn = hours + F.softplus(self.head_num(n).squeeze(-1))
            pt = hours + F.softplus(self.head_txt(t).squeeze(-1))
            w = self.mix.sigmoid()
            pred = w * pn + (1 - w) * pt
            return (pred, dict(enc=enc, num=ns, txt=ts))
        h, s = self.num(h, graphs.get("balanced") if graphs else None, state.get("num"))
        return (hours + F.softplus(self.head_num(h).squeeze(-1)), dict(enc=enc, num=s))

In [53]:
def valid_mask(ds, rows, start, stop, role):
    hours = np.arange(start + 1, stop + 1)
    return (
        (ds.role[rows, None] == ROLES[role])
        & (hours[None, :] >= 4)
        & (hours[None, :] <= ds.lengths[rows, None])
    )


def training_loss(prediction, target, mask, eligible_hours, normalizer):
    """MSE per jam, dinormalisasi agar setiap stay memiliki bobot total yang sama."""
    node, hour = torch.where(mask)
    if len(node) == 0:
        return prediction.sum() * 0
    squared_error = (prediction[node, hour] - target[node]).square()
    return (squared_error / eligible_hours[node].clamp_min(1)).sum() / normalizer


def metric_accumulate(acc, pred, y, mask, rows):
    e = pred.astype(np.float64) - y[:, None]
    a = np.where(mask, np.abs(e), 0)
    sq = np.where(mask, e * e, 0)
    acc["abs"][rows] += a.sum(1)
    acc["sq"][rows] += sq.sum(1)
    acc["n"][rows] += mask.sum(1)
    if mask.shape[1]:
        pass


def metric_finish(acc, ds, role):
    """Macro-stay MAE/RMSE/R2, using total LOS as the target in days."""
    selected = ds.role == ROLES[role]
    require(selected.any(), "Empty evaluation role")
    require(
        np.array_equal(acc["n"][selected], ds.lengths[selected] - 3),
        "Not all eligible hourly endpoints were evaluated",
    )
    per_stay_mae = acc["abs"][selected] / acc["n"][selected]
    per_stay_mse = acc["sq"][selected] / acc["n"][selected]
    mse = float(per_stay_mse.mean())
    true_days = np.asarray(ds.y[selected], dtype=np.float64)
    target_variance = float(np.mean((true_days - true_days.mean()) ** 2))
    return dict(
        MSE_days2=mse,
        MAE_days=float(per_stay_mae.mean()),
        RMSE_days=float(np.sqrt(mse)),
        R2=1 - mse / target_variance if target_variance > 0 else None,
        stays=int(selected.sum()),
        prediction_points=int(acc["n"][selected].sum()),
    )


def regression_metrics(y, prediction):
    """Ordinary metrics for one prediction per eligible stay at a landmark."""
    y = np.asarray(y, np.float64)
    prediction = np.asarray(prediction, np.float64)
    require(y.shape == prediction.shape and y.ndim == 1, "Metric shapes differ")
    require(
        np.isfinite(y).all() and np.isfinite(prediction).all(), "Nonfinite metric input"
    )
    if len(y) == 0:
        return dict(
            stays=0,
            prediction_points=0,
            MAE_days=None,
            RMSE_days=None,
            R2=None,
            MedAE_days=None,
        )
    error = prediction - y
    sse = float(np.sum(error**2))
    sst = float(np.sum((y - y.mean()) ** 2))
    return dict(
        stays=len(y),
        prediction_points=len(y),
        MAE_days=float(np.abs(error).mean()),
        RMSE_days=float(np.sqrt(np.mean(error**2))),
        R2=1 - sse / sst if sst > 0 else None,
        MedAE_days=float(np.median(np.abs(error))),
    )

In [54]:
def forward_prefix(model, ds, rows, cache, cfg, device, block_hours, total_hours):
    """Evaluation helper: preserve causal recurrent state across blocks."""
    state = None
    parts = []
    for start in range(0, total_hours, block_hours):
        stop = min(total_hours, start + block_hours)
        batch, _ = ds.block(rows, start, stop)
        graphs = (
            graph_block(cache, model.family, start, stop, cfg["k"], device)
            if cache is not None
            else None
        )
        prediction, state = model(tensor_batch(batch, device), graphs, state)
        parts.append(prediction.cpu().numpy())
    return (np.concatenate(parts, axis=1), state)


def check_evaluation_blocks(
    model, ds, groups, cache_root, cfg, device, train_block, eval_block, proof_groups=2
):
    """Sample gate for faster EVAL blocks; never increases the training block."""
    require(not model.training and (not torch.is_grad_enabled()), "Proof is eval-only")
    records = []
    for group_index in np.linspace(
        0, len(groups) - 1, min(proof_groups, len(groups)), dtype=int
    ):
        rows = groups[int(group_index)]
        total_hours = min(int(ds.lengths[rows].max()), 2 * eval_block + 7)
        cache = (
            graph_cache(
                ds,
                rows,
                Path(cache_root) / f"group_{group_index:05d}",
                GRAPH_K,
                train_block,
            )
            if model.family != "LSTM"
            else None
        )
        small, small_state = forward_prefix(
            model, ds, rows, cache, cfg, device, train_block, total_hours
        )
        large, large_state = forward_prefix(
            model, ds, rows, cache, cfg, device, eval_block, total_hours
        )
        difference = float(
            np.max(np.abs(small.astype(np.float64) - large.astype(np.float64)))
        )
        require(
            difference <= 0.0001 and np.allclose(small, large, atol=0.0001, rtol=1e-05),
            "Eval block agreement failed; set eval_block_hours = block_hours in a NEW run",
        )

        def tensors(value):
            if isinstance(value, torch.Tensor):
                return [value.detach().cpu()]
            if isinstance(value, dict):
                return [t for key in sorted(value) for t in tensors(value[key])]
            if isinstance(value, (tuple, list)):
                return [t for item in value for t in tensors(item)]
            return []

        left, right = (tensors(small_state), tensors(large_state))
        require(
            len(left) == len(right)
            and all(
                (
                    a.shape == b.shape and torch.allclose(a, b, atol=0.0001, rtol=1e-05)
                    for a, b in zip(left, right)
                )
            ),
            "Eval recurrent state agreement failed",
        )
        records.append(
            dict(
                group=int(group_index),
                hours=total_hours,
                max_abs_difference_days=difference,
                status="PASS",
            )
        )
    return records

In [55]:
def evaluate_model(
    model,
    ds,
    groups,
    cache_root,
    cfg,
    device,
    block_hours,
    until=None,
    eval_block_hours=64,
    proof_groups=2,
    return_table=False,
):
    """Evaluate ALL train/validation endpoints, with one equal weight per stay."""
    require(until is None, "Truncated evaluation is not a final metric")
    model.eval()
    roles = ("train", "validation")
    acc = {key: np.zeros(ds.n, np.float64) for key in ("abs", "sq", "n")}
    landmark = np.full((ds.n, len(LANDMARKS)), np.nan, np.float64)
    started = time.monotonic()
    last_log = started
    device_ids = [torch.device(device).index or 0] if str(device).startswith("cuda") else []
    with torch.random.fork_rng(devices=device_ids), torch.no_grad():
        requested_eval_block = eval_block_hours
        try:
            proofs = (
                check_evaluation_blocks(
                    model,
                    ds,
                    groups,
                    cache_root,
                    cfg,
                    device,
                    block_hours,
                    eval_block_hours,
                    proof_groups,
                )
                if eval_block_hours != block_hours
                else []
            )
        except ValueError as exc:
            if not str(exc).startswith(
                ("Eval block agreement failed", "Eval recurrent state agreement failed")
            ):
                raise
            eval_block_hours = block_hours
            proofs = [
                dict(
                    status="FALLBACK_TO_ORIGINAL_BLOCK",
                    reason=str(exc),
                    requested_block=requested_eval_block,
                    actual_block=block_hours,
                )
            ]
            print(
                f"Fast eval agreement failed: evaluating ALL endpoints with block={block_hours}.",
                flush=True,
            )
        for group_index, rows in enumerate(groups):
            total_hours = int(ds.lengths[rows].max())
            cache = (
                graph_cache(
                    ds,
                    rows,
                    Path(cache_root) / f"group_{group_index:05d}",
                    GRAPH_K,
                    block_hours,
                )
                if model.family != "LSTM"
                else None
            )
            state = None
            for start in range(0, total_hours, eval_block_hours):
                stop = min(total_hours, start + eval_block_hours)
                batch, _ = ds.block(rows, start, stop)
                graphs = (
                    graph_block(cache, model.family, start, stop, cfg["k"], device)
                    if cache is not None
                    else None
                )
                pred, state = model(tensor_batch(batch, device), graphs, state)
                prediction = pred.cpu().numpy().astype(np.float64)
                targets = np.zeros(len(rows), dtype=np.float64)
                allowed = np.isin(ds.role[rows], [0, 1])
                targets[allowed] = ds.y[rows[allowed]]
                require(np.isfinite(prediction).all(), "Nonfinite evaluation predictions")
                for role in roles:
                    mask = valid_mask(ds, rows, start, stop, role)
                    metric_accumulate(acc, prediction, targets, mask, rows)
                for column, hour in enumerate(LANDMARKS):
                    if start < hour <= stop:
                        eligible = (ds.lengths[rows] >= hour) & np.isin(
                            ds.role[rows], [0, 1]
                        )
                        landmark[rows[eligible], column] = prediction[
                            eligible, hour - start - 1
                        ]
                if time.monotonic() - last_log >= 25:
                    print(
                        f"Eval {group_index + 1}/{len(groups)} groups; endpoint {stop}/{total_hours}",
                        flush=True,
                    )
                    last_log = time.monotonic()
    flat = {}
    table = []
    for role in roles:
        macro = metric_finish(acc, ds, role)
        flat.update(
            {
                role + "_mse_days2": macro["MSE_days2"],
                role + "_mae_days": macro["MAE_days"],
                role + "_rmse_days": macro["RMSE_days"],
                role + "_r2": macro["R2"],
            }
        )
        table.append(
            dict(
                split=role,
                scope="all_hours_equal_stay_weight",
                estimator="model",
                endpoint_hour=None,
                MedAE_days=None,
                **macro,
            )
        )
        for column, hour in enumerate(LANDMARKS):
            eligible = (ds.role == ROLES[role]) & (ds.lengths >= hour)
            values = regression_metrics(ds.y[eligible], landmark[eligible, column])
            table.append(
                dict(
                    split=role,
                    scope="landmark",
                    estimator="model",
                    endpoint_hour=hour,
                    **values,
                )
            )
            if hour == 4:
                flat[role + "_mae_at_4h_days"] = values["MAE_days"]
    flat["evaluation_seconds"] = time.monotonic() - started
    flat["evaluation_block_hours"] = eval_block_hours
    flat["evaluation_fallback"] = eval_block_hours != requested_eval_block
    flat["evaluation_fallback_reason"] = next(
        (p["reason"] for p in proofs if "reason" in p), ""
    )
    flat["eval_proof_max_abs_difference_days"] = (
        None
        if flat["evaluation_fallback"]
        else max((p["max_abs_difference_days"] for p in proofs), default=0.0)
    )
    if return_table:
        return (flat, pd.DataFrame(table))
    return flat

In [56]:
def optimizer_and_scheduler(model, cfg, settings):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg["lr"],
        betas=(0.9, 0.999),
        weight_decay=cfg["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=settings["max_epochs"], eta_min=0.0
    )
    return (optimizer, scheduler)


def checkpoint_payload(model, optimizer, scheduler, progress, state, fingerprint):
    return dict(
        fingerprint=fingerprint,
        model=tree_map(model.state_dict(), lambda t: t.detach().cpu().clone()),
        optimizer=tree_map(optimizer.state_dict(), lambda t: t.detach().cpu().clone()),
        scheduler=copy.deepcopy(scheduler.state_dict()),
        progress=copy.deepcopy(progress),
        state=tree_map(state, lambda t: t.detach().cpu().clone()),
        rng=rng_state(),
    )


def restore_checkpoint(path, model, optimizer, scheduler, fingerprint, device):
    saved = torch.load(path, map_location="cpu", weights_only=False)
    require(
        saved["fingerprint"] == fingerprint, "Checkpoint berasal dari konfigurasi berbeda"
    )
    model.load_state_dict(saved["model"], strict=True)
    optimizer.load_state_dict(saved["optimizer"])
    scheduler.load_state_dict(saved["scheduler"])
    for state in optimizer.state.values():
        for key, value in state.items():
            if isinstance(value, torch.Tensor) and key != "step":
                state[key] = value.to(device)
    restore_rng(saved["rng"])
    return (saved["progress"], tree_map(saved["state"], lambda t: t.to(device)))

In [57]:
def train_block(
    model, ds, rows, cache, start, stop, state, optimizer, cfg, settings, device, normalizer
):
    batch, _ = ds.block(rows, start, stop)
    graphs = (
        graph_block(cache, model.family, start, stop, cfg["k"], device)
        if cache is not None
        else None
    )
    model.train()
    optimizer.zero_grad(set_to_none=True)
    prediction, next_state = model(tensor_batch(batch, device), graphs, state)
    mask = torch.as_tensor(valid_mask(ds, rows, start, stop, "train"), device=device)
    target = torch.zeros(len(rows), dtype=torch.float32, device=device)
    train_nodes = ds.role[rows] == ROLES["train"]
    target[torch.as_tensor(train_nodes, device=device)] = torch.as_tensor(
        np.array(ds.y[rows[train_nodes]], copy=True), dtype=torch.float32, device=device
    )
    hours = torch.as_tensor(ds.lengths[rows] - 3, dtype=torch.float32, device=device)
    loss = training_loss(prediction, target, mask, hours, normalizer)
    require(torch.isfinite(loss), "Training loss tidak finite")
    updated = bool(mask.any())
    if updated:
        loss.backward()
        norm = nn.utils.clip_grad_norm_(model.parameters(), settings["gradient_clip"])
        require(torch.isfinite(norm), "Gradien tidak finite")
        optimizer.step()
    return (tree_map(next_state, lambda t: t.detach()), updated)

In [58]:
def train_one_epoch(
    model,
    ds,
    groups,
    cache_root,
    cfg,
    settings,
    device,
    optimizer,
    scheduler,
    progress,
    state,
    last_path,
    fingerprint,
    interrupt_after_steps=None,
):
    """TBPTT: state diteruskan antarblok, lalu dilepas dari grafik gradien."""
    order = np.random.default_rng(
        np.random.SeedSequence([settings["seed"], progress["epoch"]])
    ).permutation(len(groups))
    normalizer = int((ds.role == ROLES["train"]).sum()) / len(groups)
    saved_at = logged_at = time.monotonic()
    initial_steps = progress["steps"]
    for position in range(progress["group_position"], len(order)):
        group_index = int(order[position])
        rows = groups[group_index]
        total = int(ds.lengths[rows].max())
        cache = (
            graph_cache(
                ds,
                rows,
                Path(cache_root) / f"group_{group_index:05d}",
                GRAPH_K,
                settings["block_hours"],
            )
            if model.family != "LSTM"
            else None
        )
        offset = progress["offset"] if position == progress["group_position"] else 0
        if offset == 0:
            state = None
        for start in range(offset, total, settings["block_hours"]):
            stop = min(total, start + settings["block_hours"])
            state, updated = train_block(
                model,
                ds,
                rows,
                cache,
                start,
                stop,
                state,
                optimizer,
                cfg,
                settings,
                device,
                normalizer,
            )
            progress["steps"] += int(updated)
            if stop == total:
                progress.update(group_position=position + 1, offset=0)
                state = None
            else:
                progress.update(group_position=position, offset=stop)
            interrupt = (
                interrupt_after_steps
                and progress["steps"] - initial_steps >= interrupt_after_steps
            )
            if time.monotonic() - saved_at >= settings["save_seconds"] or interrupt:
                atomic_torch(
                    last_path,
                    checkpoint_payload(
                        model, optimizer, scheduler, progress, state, fingerprint
                    ),
                )
                saved_at = time.monotonic()
            if interrupt:
                raise KeyboardInterrupt()
            if time.monotonic() - logged_at >= 25:
                print(
                    f"{model.family}/{model.fusion}/{cfg['name']}: epoch {progress['epoch'] + 1}, group {position + 1}/{len(groups)}, hour {stop}/{total}, steps {progress['steps']:,}",
                    flush=True,
                )
                logged_at = time.monotonic()
    return state

In [59]:
def select_best_checkpoint(
    model, progress, metrics, cfg, ds, settings, best_path, fingerprint
):
    score = metrics["validation_mse_days2"]
    improved = (
        progress["best_mse"] is None or score < progress["best_mse"] - settings["min_delta"]
    )
    if improved:
        progress.update(best_mse=score, best_epoch=progress["epoch"] + 1, bad_epochs=0)
        atomic_torch(
            best_path,
            dict(
                fingerprint=fingerprint,
                model=model.state_dict(),
                cfg=cfg,
                family=model.family,
                fusion=model.fusion,
                static_dim=ds.S,
                attention_window=settings["attention_window"],
                selected_epoch=progress["best_epoch"],
                best_validation_mse_days2=score,
            ),
        )
    else:
        progress["bad_epochs"] += 1

In [60]:
def train_trial(
    ds,
    groups,
    cache_root,
    folder,
    report,
    family,
    fusion,
    cfg,
    execution,
    settings,
    fingerprint,
    interrupt_after_steps=None,
):
    folder, report = (Path(folder), Path(report))
    folder.mkdir(parents=True, exist_ok=True)
    report.mkdir(parents=True, exist_ok=True)
    device = execution["device"]
    torch.set_num_threads(execution["threads"])
    seed_all(settings["seed"])
    model = LOSModel(ds.S, family, fusion, cfg, settings["attention_window"]).to(device)
    optimizer, scheduler = optimizer_and_scheduler(model, cfg, settings)
    last_path, best_path, completed_path = (
        folder / name for name in ["last_LOCAL.pt", "best_LOCAL.pt", "complete.json"]
    )
    if completed_path.exists():
        result = read_json(completed_path)
        require(
            result["fingerprint"] == fingerprint
            and sha(best_path) == result["best_sha256"],
            "Hasil trial selesai berubah",
        )
        return result
    progress = dict(
        epoch=0,
        group_position=0,
        offset=0,
        steps=0,
        best_mse=None,
        best_epoch=None,
        bad_epochs=0,
        history=[],
    )
    state = None
    if last_path.exists():
        progress, state = restore_checkpoint(
            last_path, model, optimizer, scheduler, fingerprint, device
        )
        print(
            f"Resume {family}/{fusion}/{cfg['name']}: epoch {progress['epoch'] + 1}, step {progress['steps']:,}"
        )
    while (
        progress["epoch"] < settings["max_epochs"]
        and progress["bad_epochs"] < settings["patience"]
    ):
        started = time.monotonic()
        learning_rate_used = optimizer.param_groups[0]["lr"]
        state = train_one_epoch(
            model,
            ds,
            groups,
            cache_root,
            cfg,
            settings,
            device,
            optimizer,
            scheduler,
            progress,
            state,
            last_path,
            fingerprint,
            interrupt_after_steps,
        )
        metrics = evaluate_model(
            model,
            ds,
            groups,
            cache_root,
            cfg,
            device,
            settings["block_hours"],
            eval_block_hours=settings["eval_block_hours"],
            proof_groups=settings["proof_groups"],
        )
        select_best_checkpoint(
            model, progress, metrics, cfg, ds, settings, best_path, fingerprint
        )
        progress["history"].append(
            dict(
                epoch=progress["epoch"] + 1,
                optimizer_steps=progress["steps"],
                **metrics,
                best_epoch=progress["best_epoch"],
                best_validation_mse_days2=progress["best_mse"],
                epochs_without_improvement=progress["bad_epochs"],
                learning_rate=learning_rate_used,
                epoch_seconds_this_session=time.monotonic() - started,
            )
        )
        scheduler.step()
        progress.update(epoch=progress["epoch"] + 1, group_position=0, offset=0)
        state = None
        atomic_torch(
            last_path,
            checkpoint_payload(model, optimizer, scheduler, progress, state, fingerprint),
        )
        pd.DataFrame(progress["history"]).to_csv(report / "learning_curve.csv", index=False)
        plot_learning_curves(report)
        print(
            f"{family}/{fusion}/{cfg['name']}: epoch {progress['epoch']}; validation MSE={metrics['validation_mse_days2']:.5f}; MAE={metrics['validation_mae_days']:.5f}; RMSE={metrics['validation_rmse_days']:.5f}; R2={metrics['validation_r2']}; best epoch={progress['best_epoch']}; patience {progress['bad_epochs']}/{settings['patience']}",
            flush=True,
        )
    pd.DataFrame(progress["history"]).to_csv(report / "learning_curve.csv", index=False)
    plot_learning_curves(report)
    best = torch.load(best_path, map_location="cpu", weights_only=False)
    model.load_state_dict(best["model"], strict=True)
    metrics, table = evaluate_model(
        model,
        ds,
        groups,
        cache_root,
        cfg,
        device,
        settings["block_hours"],
        eval_block_hours=settings["eval_block_hours"],
        proof_groups=settings["proof_groups"],
        return_table=True,
    )
    require(
        abs(metrics["validation_mse_days2"] - progress["best_mse"]) <= 0.0001,
        "Evaluasi ulang best checkpoint tidak cocok",
    )
    table.insert(0, "selected_epoch", progress["best_epoch"])
    table.to_csv(report / "best_validation_metrics.csv", index=False)
    result = dict(
        status="COMPLETE",
        fingerprint=fingerprint,
        family=family,
        fusion=fusion,
        cfg=cfg,
        best_epoch=progress["best_epoch"],
        best_validation_mse_days2=progress["best_mse"],
        best_validation_mae_days=metrics["validation_mae_days"],
        best_validation_rmse_days=metrics["validation_rmse_days"],
        best_validation_r2=metrics["validation_r2"],
        best_checkpoint=str(best_path.resolve()),
        best_sha256=sha(best_path),
        epochs=progress["epoch"],
        stop_reason=(
            "EARLY_STOPPING"
            if progress["bad_epochs"] >= settings["patience"]
            else "MAX_EPOCHS"
        ),
        test_evaluated=False,
    )
    atomic_json(completed_path, result)
    return result

In [61]:
def plot_learning_curves(folder):
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MultipleLocator, FuncFormatter

    folder = Path(folder)
    table = pd.read_csv(folder / "learning_curve.csv")
    fig, axes = plt.subplots(1, 4, figsize=(19, 4))
    for axis, metric, label in zip(
        axes,
        ["mse_days2", "mae_days", "rmse_days", "r2"],
        ["MSE (hari²)", "MAE (hari)", "RMSE (hari)", "R²"],
    ):
        axis.plot(table.epoch, table["train_" + metric], label="Train", color="#2563eb")
        axis.plot(
            table.epoch, table["validation_" + metric], label="Validation", color="#f97316"
        )
        axis.axvline(
            int(table.iloc[-1].best_epoch),
            color="#64748b",
            linestyle="--",
            label="Best validation MSE",
        )
        axis.set(xlabel="Epoch", ylabel=label, title=label)
        axis.grid(alpha=0.25)
        if metric == "r2":
            axis.yaxis.set_major_locator(MultipleLocator(0.1))
            axis.yaxis.set_major_formatter(
                FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ","))
            )
        axis.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(folder / "learning_curves.png", dpi=160)
    plt.close(fig)

In [62]:
def trial_definitions():
    trials = []
    for family in FAMILIES:
        for fusion in FUSIONS:
            for cfg in MODEL_CONFIGS[family]:
                trials.append(
                    dict(
                        name=family + "_" + fusion + "_" + cfg["name"],
                        family=family,
                        fusion=fusion,
                        cfg=dict(cfg),
                    )
                )
    return trials


def select_trials(all_trials, names):
    if not names:
        return all_trials
    valid = {trial["name"] for trial in all_trials}
    require(set(names).issubset(valid), "TRIALS_TO_RUN berisi nama yang tidak tersedia")
    return [trial for trial in all_trials if trial["name"] in names]


def validate_settings(settings):
    require(
        settings["group_size"] > GRAPH_K
        and settings["block_hours"] > 0
        and (settings["attention_window"] > 0)
        and (settings["eval_block_hours"] > 0)
        and (settings["proof_groups"] > 0)
        and (settings["max_epochs"] > 0)
        and (settings["patience"] > 0)
        and (settings["min_delta"] == 0.0)
        and (settings["save_seconds"] > 0)
        and (settings["threads"] > 0),
        "Pengaturan runtime tidak valid",
    )
    for trial in trial_definitions():
        cfg = trial["cfg"]
        require(
            cfg["loss"] == "mse" and cfg["lr"] > 0 and (cfg["d_model"] == 64),
            "Konfigurasi model berbeda",
        )
        if trial["family"] != "LSTM":
            require(
                cfg["heads"] == 8 and cfg["graph_layers"] == 4 and (cfg["k"] == GRAPH_K),
                "Konfigurasi graf berbeda",
            )

In [63]:
def benchmark_models(ds, groups, folder, settings, trials, devices=None):
    """Ukur beberapa update model sementara; tidak menghasilkan checkpoint training."""
    rows = next((rows for rows in groups if (ds.role[rows] == 0).any()))
    stop = min(max(4, settings["block_hours"]), int(ds.lengths[rows].max()))
    graph = graph_cache(
        ds, rows, Path(folder) / "probe", GRAPH_K, settings["block_hours"], until=stop
    )
    batch, _ = ds.block(rows, 0, stop)
    options = devices or [("cpu", settings["threads"])] + (
        [("cuda:0", 1)] if torch.cuda.is_available() else []
    )
    records, choices = ([], {})
    for trial in trials:
        candidates = []
        for device, threads in options:
            try:
                seed_all(settings["seed"])
                torch.set_num_threads(threads)
                model = LOSModel(
                    ds.S,
                    trial["family"],
                    trial["fusion"],
                    trial["cfg"],
                    settings["attention_window"],
                ).to(device)
                optimizer, _ = optimizer_and_scheduler(model, trial["cfg"], settings)
                graphs = (
                    graph_block(graph, trial["family"], 0, stop, GRAPH_K, device)
                    if trial["family"] != "LSTM"
                    else None
                )
                timings = []
                for repeat in range(3):
                    if device.startswith("cuda"):
                        torch.cuda.synchronize()
                        torch.cuda.reset_peak_memory_stats()
                    started = time.perf_counter()
                    state, updated = train_block(
                        model,
                        ds,
                        rows,
                        graph if graphs is not None else None,
                        0,
                        stop,
                        None,
                        optimizer,
                        trial["cfg"],
                        settings,
                        device,
                        max(1, int((ds.role[rows] == 0).sum())),
                    )
                    require(updated, "Probe harus memiliki target train")
                    if device.startswith("cuda"):
                        torch.cuda.synchronize()
                    timings.append(time.perf_counter() - started)
                row = dict(
                    trial=trial["name"],
                    device=device,
                    threads=threads,
                    status="PASS",
                    seconds=float(np.median(timings[1:])),
                    sampled_hours=stop,
                    nodes=len(rows),
                    peak_cuda_gib=(
                        torch.cuda.max_memory_allocated() / 2**30
                        if device.startswith("cuda")
                        else None
                    ),
                )
                candidates.append(row)
                del model, optimizer, state
            except (RuntimeError, MemoryError) as error:
                row = dict(
                    trial=trial["name"],
                    device=device,
                    threads=threads,
                    status="FAILED",
                    reason=str(error),
                )
            records.append(row)
            if device.startswith("cuda"):
                torch.cuda.empty_cache()
        require(candidates, "Tidak ada device yang lolos untuk " + trial["name"])
        best = min(candidates, key=lambda record: record["seconds"])
        choices[trial["name"]] = dict(device=best["device"], threads=best["threads"])
        print("Benchmark:", trial["name"], choices[trial["name"]], flush=True)
    return (pd.DataFrame(records), choices)

In [64]:
def synthetic_checks():
    """No clinical files. Exercise all nine actual model implementations."""
    torch.set_num_threads(1)
    seed_all(91)
    N, L, S = (5, 11, 7)
    cfg = dict(MODEL_CONFIGS["DGT"][0], d_model=8, heads=8, dropout=0.0, k=2)
    b = dict(
        static=torch.randn(N, S),
        numeric=torch.randn(N, L, 43),
        numeric_mask=torch.ones(N, L, 43),
        notes=torch.randn(N, L, 768),
        notes_mask=torch.ones(N, L),
        diagnosis=torch.randn(N, 768),
        diagnosis_mask=torch.ones(N, L),
        hours=torch.arange(1, L + 1).float(),
    )
    b["diagnosis_mask"][:, :4] = 0
    b["notes_mask"][:, :3] = 0
    e = torch.tensor([[[1, 2], [0, 2], [0, 3], [1, 4], [2, 3]]] * L)
    w = torch.ones_like(e).float()
    graphs = {v: (e, w) for v in ["balanced", "numeric", "text"]}
    checks = []
    for family in FAMILIES:
        for fusion in FUSIONS:
            seed_all(91)
            model = LOSModel(S, family, fusion, cfg, 4).eval()
            if family != "LSTM":
                for branch in [model.num, model.txt] if fusion == "late" else [model.num]:
                    require(
                        len(branch.graph) == 4
                        and all((layer.heads == 8 for layer in branch.graph))
                        and (branch.attn.num_heads == 8),
                        "Four graph layers/eight heads failed",
                    )
            else:
                require(
                    not hasattr(model.num, "graph"),
                    "LSTM baseline unexpectedly has graph attention",
                )
            g = graphs if family != "LSTM" else None
            p, _ = model(b, g)
            require(
                p.shape == (N, L)
                and torch.isfinite(p).all()
                and (p >= b["hours"] / 24).all(),
                "Shape/physical head failed",
            )
            changed = {k: v.clone() for k, v in b.items()}
            changed["numeric"][:, 7:] += 100
            changed["notes"][:, 7:] -= 90
            changed["diagnosis"] = b["diagnosis"].clone()
            q, _ = model(changed, g)
            require(
                torch.allclose(p[:, :7], q[:, :7], atol=2e-06, rtol=1e-05),
                "Future input changed past output",
            )
            gate = {k: v.clone() for k, v in b.items()}
            gate["diagnosis"] += 77
            gate["notes"][:, :3] += 99
            q, _ = model(gate, g)
            require(
                torch.allclose(p[:, :3], q[:, :3], atol=2e-06, rtol=1e-05),
                "Unavailable text gate failed",
            )
            state = None
            parts = []
            for a, z in [(0, 3), (3, 7), (7, L)]:
                bb = {
                    k: (
                        v
                        if k in ["static", "diagnosis"]
                        else v[a:z] if k == "hours" else v[:, a:z]
                    )
                    for k, v in b.items()
                }
                gg = (
                    {v: (edge[a:z], weight[a:z]) for v, (edge, weight) in graphs.items()}
                    if g
                    else None
                )
                pp, state = model(bb, gg, state)
                parts.append(pp)
            require(
                torch.allclose(p, torch.cat(parts, 1), atol=2e-06, rtol=1e-05),
                "Chunked/full causal execution differs",
            )
            mask = torch.zeros(N, L, dtype=torch.bool)
            mask[:3, 3:] = True
            y = torch.arange(N).float() + 2
            loss = training_loss(p, y, mask, torch.full((N,), 8.0), 3)
            altered = y.clone()
            altered[3:] = float("nan")
            require(
                torch.equal(
                    loss, training_loss(p, altered, mask, torch.full((N,), 8.0), 3)
                ),
                "Held-out labels leaked into loss",
            )
            loss.backward()
            require(
                all(
                    (
                        torch.isfinite(x.grad).all()
                        for x in model.parameters()
                        if x.grad is not None
                    )
                ),
                "Nonfinite gradients",
            )
            checks.append(
                dict(
                    check=family
                    + "_"
                    + fusion
                    + "_causality_gates_streaming_loss_gradients",
                    status="PASS",
                )
            )
    num = np.eye(5, dtype=np.float32)
    txt = np.roll(num, 1, axis=0)
    subjects = np.array([1, 1, 2, 3, 4])
    edges, _ = topk_graph(num, txt, np.ones(5, bool), subjects, 4, "balanced")
    for i in range(5):
        require(
            all((subjects[i] != subjects[j] for j in edges[i] if j >= 0)),
            "Same-patient graph edge",
        )
    fake = {"balanced": (np.tile(edges[None], (L, 1, 1)), np.ones((L, 5, 4), np.float32))}
    ee, _ = graph_block(fake, "StaticGraphTransformer", 0, L, 2, "cpu")["balanced"]
    require(
        (ee[:3] == -1).all() and torch.equal(ee[3], ee[-1]),
        "Static graph warm-up/fixed topology failed",
    )
    require(
        np.array_equal(
            np.concatenate(fixed_groups(23, 6, 42)),
            np.random.default_rng(42).permutation(23),
        ),
        "Grouping not ID/target independent",
    )
    checks.append(
        dict(
            check="same_patient_exclusion_fixed_static_topology_and_outcome_independent_grouping",
            status="PASS",
        )
    )
    require(
        all(
            (
                c["heads"] == 8 and c["graph_layers"] == 4 and (c["d_model"] % 8 == 0)
                for family, configs in MODEL_CONFIGS.items()
                if family != "LSTM"
                for c in configs
            )
        ),
        "Production candidate architecture differs",
    )
    checks.append(
        dict(
            check="all_graph_fusion_branches_four_layers_eight_heads_and_LSTM_no_graph",
            status="PASS",
        )
    )
    seed_all(21)
    m = nn.Sequential(nn.Linear(3, 4), nn.Dropout(0.2), nn.Linear(4, 1))
    o = torch.optim.Adam(m.parameters(), lr=0.01)
    x = torch.ones(3, 3)

    def update():
        o.zero_grad()
        z = m(x).square().mean()
        z.backward()
        o.step()

    update()
    saved = copy.deepcopy(dict(model=m.state_dict(), opt=o.state_dict(), rng=rng_state()))
    update()
    expected = copy.deepcopy(m.state_dict())
    m.load_state_dict(saved["model"])
    o.load_state_dict(saved["opt"])
    restore_rng(saved["rng"])
    update()
    require(
        all((torch.equal(v, m.state_dict()[k]) for k, v in expected.items())),
        "Optimizer/RNG continuation failed",
    )
    checks.append(dict(check="optimizer_dropout_RNG_next_update_reproduced", status="PASS"))
    prediction = torch.tensor([[1.0, 3.0, 9.0], [2.0, 7.0, 4.0]], requires_grad=True)
    target = torch.tensor([2.0, 4.0])
    mask = torch.tensor([[True, True, False], [True, False, False]])
    loss = training_loss(prediction, target, mask, torch.tensor([2.0, 1.0]), 2)
    require(torch.equal(loss, torch.tensor(2.5)), "Equal-stay MSE equation failed")
    loss.backward()
    require((prediction.grad[~mask] == 0).all(), "Masked loss has gradients")
    checks.append(
        dict(check="independent_equal_stay_MSE_and_masked_gradients", status="PASS")
    )
    require(
        len(trial_definitions()) == 12
        and len({t["name"] for t in trial_definitions()}) == 12,
        "Expected twelve distinct trials",
    )
    checks.append(dict(check="twelve_family_fusion_learning_rate_trials", status="PASS"))
    return pd.DataFrame(checks)

In [65]:
def prepare_experiment(root, stage03_run, stage04_run, resume_run, settings):
    run_id = (
        resume_run
        or datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f") + "_models_70_15_15"
    )
    require(
        Path(run_id).name == run_id and "/" not in run_id and ("\\" not in run_id),
        "Nama run invalid",
    )
    require(run_id not in [stage03_run, stage04_run], "Run model harus berbeda dari input")
    data, report = [Path(root) / parent / run_id for parent in ["data", "outputs"]]
    data.mkdir(parents=True, exist_ok=True)
    report.mkdir(parents=True, exist_ok=True)
    return (run_id, data, report)


def experiment_contract(inputs, settings, groups, trials):
    return dict(
        version="icu_los_holdout_models_mse_v1",
        code_sha256=runtime_code_hash(),
        cohort_sha256=inputs["cohort_sha256"],
        settings=settings,
        trials=trials,
        roles=ROLES,
        landmarks=list(LANDMARKS),
        graph_k=GRAPH_K,
        group_membership_sha256=digest([rows.tolist() for rows in groups]),
        loss="MSE macro-stay",
        selection="validation_mse_days2",
        source_hashes={p: a["sha256"] for p, a in inputs["watch"].items()},
        environment=dict(
            python=platform.python_version(),
            numpy=np.__version__,
            pandas=pd.__version__,
            torch=str(torch.__version__),
            cuda=torch.version.cuda,
            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        ),
    )


def watch_unchanged(watch):
    for p, a in watch.items():
        st = Path(p).stat()
        require(
            st.st_size == a["bytes"] and st.st_mtime_ns == a["mtime_ns"],
            "Source changed during execution: " + Path(p).name,
        )

In [66]:
def save_validation_selection(report, results):
    table = pd.DataFrame(
        [
            {
                key: result[key]
                for key in [
                    "family",
                    "fusion",
                    "best_epoch",
                    "best_validation_mse_days2",
                    "best_validation_mae_days",
                    "best_validation_rmse_days",
                    "best_validation_r2",
                    "best_checkpoint",
                ]
            }
            | {"candidate": result["cfg"]["name"]}
            for result in results
        ]
    )
    table.to_csv(Path(report) / "validation_results.csv", index=False)
    selected = (
        table.sort_values(["best_validation_mse_days2", "candidate"])
        .groupby(["family", "fusion"], sort=False)
        .head(1)
    )
    selected.to_csv(Path(report) / "selected_candidates.csv", index=False)
    return selected


def evaluate_saved_trial(
    ds, groups, data, report, trial, execution, settings, base_fingerprint
):
    name, cfg = (trial["name"], trial["cfg"])
    folder = Path(data) / "trials" / name
    path = folder / "best_LOCAL.pt"
    require(path.is_file(), "Best checkpoint belum ada: " + name)
    saved = torch.load(path, map_location="cpu", weights_only=False)
    expected = digest(dict(base=base_fingerprint, trial=trial, execution=execution))
    require(saved["fingerprint"] == expected, "Fingerprint checkpoint berbeda")
    torch.set_num_threads(execution["threads"])
    model = LOSModel(
        ds.S, trial["family"], trial["fusion"], cfg, settings["attention_window"]
    ).to(execution["device"])
    model.load_state_dict(saved["model"], strict=True)
    _, table = evaluate_model(
        model,
        ds,
        groups,
        Path(data) / "graph_cache",
        cfg,
        execution["device"],
        settings["block_hours"],
        eval_block_hours=settings["eval_block_hours"],
        proof_groups=settings["proof_groups"],
        return_table=True,
    )
    table.insert(0, "selected_epoch", saved["selected_epoch"])
    output = (
        Path(report)
        / "evaluation"
        / name
        / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    )
    output.mkdir(parents=True)
    table.to_csv(output / "metrics.csv", index=False)
    return dict(report_directory=str(output.resolve()), test_scored=False)

In [67]:
def run_stage05(
    root,
    stage03_run,
    stage04_run,
    mode,
    resume_run,
    settings,
    trial_names=None,
    evaluation_trial=None,
):
    require(
        mode in ["benchmark", "train", "evaluate"],
        "MODE harus benchmark, train, atau evaluate",
    )
    require(
        mode == "benchmark" or resume_run,
        "Isi RESUME_RUN dengan run benchmark notebook ini",
    )
    validate_settings(settings)
    run_id, data, report = prepare_experiment(
        root, stage03_run, stage04_run, resume_run, settings
    )
    with run_lock(data / "run.lock"):
        inputs = verify_sources(root, stage03_run, stage04_run, data)
        ds = HoldoutData(inputs)
        try:
            groups = fixed_groups(ds.n, settings["group_size"], settings["seed"])
            trials = trial_definitions()
            selected = select_trials(trials, trial_names)
            contract = experiment_contract(inputs, settings, groups, trials)
            fingerprint = digest(contract)
            contract_path = report / "model_contract.json"
            if contract_path.exists():
                require(
                    read_json(contract_path) == contract,
                    "Konfigurasi/sumber/kode berbeda; buat run baru",
                )
            else:
                require(mode == "benchmark", "Benchmark belum tersedia")
                atomic_json(contract_path, contract)
                atomic_json(
                    data / "graph_groups_LOCAL.json", [rows.tolist() for rows in groups]
                )
            self_checks = synthetic_checks()
            self_checks = pd.concat(
                [
                    self_checks,
                    pd.DataFrame(
                        {
                            "check": [
                                "holdout_upstream_hashes_and_patient_admission_isolation",
                                "fixed_groups_cover_each_cohort_row_once",
                            ],
                            "status": "PASS",
                        }
                    ),
                ],
                ignore_index=True,
            )
            require(
                np.array_equal(np.sort(np.concatenate(groups)), np.arange(ds.n)),
                "Group registry tidak lengkap",
            )
            self_checks.to_csv(report / "quality_checks.csv", index=False)
            metadata = dict(
                run_id=run_id,
                stage="05_holdout_models",
                fingerprint=fingerprint,
                stage03_run=stage03_run,
                stage04_run=stage04_run,
                settings=settings,
                data_directory=str(data.resolve()),
                report_directory=str(report.resolve()),
                cohort_stays=ds.n,
                total_trials=len(trials),
                test_scored=False,
            )
            if mode == "benchmark":
                table, choices = benchmark_models(
                    ds, groups, data / "benchmark_graphs", settings, trials
                )
                table.to_csv(report / "speed_benchmark.csv", index=False)
                atomic_json(report / "benchmark_choices.json", choices)
                atomic_json(
                    report / "benchmark_receipt.json",
                    dict(
                        fingerprint=fingerprint,
                        choices_sha256=sha(report / "benchmark_choices.json"),
                    ),
                )
                metadata["status"] = "BENCHMARK_COMPLETE_READY_FOR_TRAINING"
            else:
                receipt = read_json(report / "benchmark_receipt.json")
                require(
                    receipt["fingerprint"] == fingerprint
                    and receipt["choices_sha256"] == sha(report / "benchmark_choices.json"),
                    "Benchmark receipt berubah",
                )
                choices = read_json(report / "benchmark_choices.json")
                if mode == "evaluate":
                    require(
                        evaluation_trial in {trial["name"] for trial in trials},
                        "EVALUATION_TRIAL tidak valid",
                    )
                    trial = next(
                        (trial for trial in trials if trial["name"] == evaluation_trial)
                    )
                    result = evaluate_saved_trial(
                        ds,
                        groups,
                        data,
                        report,
                        trial,
                        choices[trial["name"]],
                        settings,
                        fingerprint,
                    )
                    watch_unchanged(inputs["watch"])
                    return result
                metadata["status"] = "TRAINING"
                atomic_json(report / "run_metadata.json", metadata)
                results = []
                for trial in selected:
                    name = trial["name"]
                    execution = choices[name]
                    tf = digest(dict(base=fingerprint, trial=trial, execution=execution))
                    result = train_trial(
                        ds,
                        groups,
                        data / "graph_cache",
                        data / "trials" / name,
                        report / "trials" / name,
                        trial["family"],
                        trial["fusion"],
                        trial["cfg"],
                        execution,
                        settings,
                        tf,
                    )
                    results.append(result)
                    save_validation_selection(report, results)
                metadata.update(
                    status=(
                        "TRAINING_COMPLETE_NO_TEST_SCORING"
                        if len(results) == len(trials)
                        else "SELECTED_TRIALS_COMPLETE_NO_TEST_SCORING"
                    ),
                    completed_trials=len(results),
                )
            watch_unchanged(inputs["watch"])
            atomic_json(report / "run_metadata.json", metadata)
            summary = "\n".join(
                [
                    "ICU LOS — STEP05 HOLDOUT MODELS",
                    "Run: " + run_id,
                    "Status: " + metadata["status"],
                    f"Cohort stays: {ds.n:,}",
                    "Order: LSTM, SGT, DGT; early/intermediate/late; 12 configured trials.",
                    "Loss: equal-stay MSE; checkpoint and early stopping: validation MSE.",
                    f"Max epochs: {settings['max_epochs']}; patience: {settings['patience']}; Adam; cosine annealing per epoch.",
                    "SGT/DGT: four graph layers, eight heads, at most ten external neighbours.",
                    "SGT/DGT temporal backbone: LSTM plus causal local attention; LSTM baseline has no graph.",
                    "Transductive fixed subgraphs; neighbour candidates limited to each group; only train targets backpropagate.",
                    "No test scoring; no BERT computation; reports: " + str(report),
                ]
            )
            (report / "summary.txt").write_text(summary, encoding="utf-8")
            print(summary)
            return metadata
        finally:
            ds.close()

In [68]:
ENGINE_NAMES = [
    "require",
    "read_json",
    "atomic_json",
    "sha",
    "artifact",
    "atomic_torch",
    "atomic_npy",
    "run_lock",
    "seed_all",
    "rng_state",
    "restore_rng",
    "tree_map",
    "fixed_groups",
    "normalize",
    "topk_graph",
    "graph_cache",
    "graph_block",
    "tensor_batch",
    "SpatialAttention",
    "Backbone",
    "FusionEncoder",
    "LOSModel",
    "valid_mask",
    "metric_accumulate",
    "metric_finish",
    "regression_metrics",
    "forward_prefix",
    "check_evaluation_blocks",
    "evaluate_model",
    "watch_unchanged",
    "HoldoutData",
    "training_loss",
    "optimizer_and_scheduler",
    "checkpoint_payload",
    "restore_checkpoint",
    "train_block",
    "train_one_epoch",
    "select_best_checkpoint",
    "train_trial",
    "plot_learning_curves",
    "verify_sources",
    "runtime_code_hash",
    "digest",
    "trial_definitions",
    "select_trials",
    "benchmark_models",
    "prepare_experiment",
    "experiment_contract",
    "save_validation_selection",
    "evaluate_saved_trial",
    "run_stage05",
    "validate_settings",
    "synthetic_checks",
]

In [69]:
FAST_INPUTS = True

# Definisi asli tetap dipakai untuk memeriksa kontrak run lama.
ReferenceData = HoldoutData
reference_hash = runtime_code_hash
reference_lock = run_lock


def prepare_fast_index(ds):
    """Metadata kecil di RAM; fitur dan embedding tetap memakai file aslinya."""
    ds._fast_partition = ds.seq.partition.to_numpy(dtype=np.int64)
    ds._fast_local = ds.seq.local_stay_row.to_numpy(dtype=np.int64)
    t = ds.temporal_params
    ds._fast_volume_fill = np.asarray(
        [(t['median'][q] - t['mean'][q]) / t['scale'][q] for q in range(30, 43)],
        dtype=np.float32,
    )


def fast_input_block(ds, rows, start, stop):
    """Input identik: salin kolom sekaligus dan cari interval catatan dengan indeks."""
    rows = np.asarray(rows, dtype=np.int64)
    n, length = len(rows), stop - start
    hours = np.arange(start + 1, stop + 1, dtype=np.float32)
    positions = np.arange(start, stop)
    numeric = np.empty((n, length, 43), dtype=np.float32)
    numeric_mask = np.zeros_like(numeric)
    notes = np.zeros((n, length, 768), dtype=np.float32)
    notes_mask = np.zeros((n, length), dtype=np.float32)
    counts = np.zeros((n, length), dtype=np.int32)
    index = ds.text['note_interval_index_LOCAL.npy']
    offsets = ds.text['note_stay_offsets_LOCAL.npy']

    for j, row in enumerate(rows):
        _, offset, observed, values = ds.partition(int(ds._fast_partition[row]))
        local = int(ds._fast_local[row])
        first, end = int(offset[local]), int(offset[local + 1])
        stay_length = end - first
        # Pengukuran memakai nilai terakhir hanya untuk padding yang dimask.
        source_rows = first + np.minimum(positions, stay_length - 1)
        numeric[j, :, :30] = values[source_rows, :30]
        numeric[j, :, 30:] = ds._fast_volume_fill
        available = max(0, min(stop, stay_length) - start)
        if available:
            source = slice(first + start, first + start + available)
            numeric[j, :available, 30:] = values[source, 30:]
            numeric_mask[j, :available] = observed[source]

        lo, hi = int(offsets[row]), int(offsets[row + 1])
        note_hours = index[lo:hi, 1]
        left = lo + int(np.searchsorted(note_hours, start, side='left'))
        right = lo + int(np.searchsorted(note_hours, stop, side='left'))
        if right > left:
            note_positions = index[left:right, 1].astype(np.int64) - start
            notes[j, note_positions] = ds.text['note_interval_vectors_LOCAL.npy'][left:right]
            notes_mask[j, note_positions] = 1
            counts[j, note_positions] = index[left:right, 2]

    present = np.asarray(ds.text['diagnosis_present_LOCAL.npy'][rows], dtype=np.float32)
    available_hour = np.asarray(
        ds.text['diagnosis_available_hour_LOCAL.npy'][rows], dtype=np.float64
    )
    diagnosis_mask = (
        (hours[None, :].astype(np.float64) > available_hour[:, None])
        & present[:, None].astype(bool)
    ).astype(np.float32)
    batch = dict(
        static=ds.static[rows].copy(),
        numeric=numeric,
        numeric_mask=numeric_mask,
        notes=notes,
        notes_mask=notes_mask,
        diagnosis=np.asarray(ds.text['diagnosis_vectors_LOCAL.npy'][rows], dtype=np.float32).copy(),
        diagnosis_mask=diagnosis_mask,
        hours=hours,
    )
    require(all(np.isfinite(a).all() for a in batch.values()), 'Model input tidak finite')
    return batch, counts


def identical_input_blocks(expected, actual):
    a, ac = expected
    b, bc = actual
    return (
        a.keys() == b.keys()
        and all(a[k].dtype == b[k].dtype and np.array_equal(a[k], b[k]) for k in a)
        and ac.dtype == bc.dtype and np.array_equal(ac, bc)
    )


class FastInputData(ReferenceData):
    """Loader cepat dengan uji kesamaan penuh pada setiap kelompok yang ditemui."""

    def __init__(self, inputs):
        super().__init__(inputs)
        prepare_fast_index(self)
        self._input_verified = set()
        self._fast_event = None

    def block(self, rows, start, stop):
        key = tuple(np.asarray(rows, dtype=np.int64).tolist())
        if key not in self._input_verified:
            # Periksa blok saat ini, awal, serta batas stay terpendek dan terpanjang.
            durations = self.lengths[rows]
            starts = sorted(set([start, 0, 3, max(0, int(durations.min()) - 1),
                                 max(0, int(durations.max()) - 1)]))
            width = max(1, stop - start)
            before = rng_state()
            reference_seconds = fast_seconds = 0.0
            for position in starts:
                end = position + width
                began = time.perf_counter()
                expected = ReferenceData.block(self, rows, position, end)
                reference_seconds += time.perf_counter() - began
                began = time.perf_counter()
                actual = fast_input_block(self, rows, position, end)
                fast_seconds += time.perf_counter() - began
                require(identical_input_blocks(expected, actual),
                        'Input loader berbeda; percepatan ditolak')
            # Loader tidak menggunakan RNG. Pengujian juga tidak mengubah RNG training.
            restore_rng(before)
            self._input_verified.add(key)
            if self._fast_event:
                self._fast_event(dict(
                    event='INPUT_GROUP_EXACT_PASS', rows=len(rows), probes=len(starts),
                    original_seconds=reference_seconds, fast_seconds=fast_seconds,
                    exact_values_masks_dtypes_and_counts=True,
                ))
        return fast_input_block(self, rows, start, stop)


def fast_extension_hash():
    """Identitas implementasi tambahan, terpisah dari fingerprint engine asli."""
    import types

    def encode(value):
        if isinstance(value, types.CodeType):
            return dict(code=value.co_code.hex(), constants=[encode(x) for x in value.co_consts],
                        names=value.co_names, variables=value.co_varnames,
                        free=value.co_freevars, cells=value.co_cellvars, flags=value.co_flags,
                        arguments=[value.co_argcount, value.co_posonlyargcount, value.co_kwonlyargcount],
                        exceptions=value.co_exceptiontable.hex())
        if isinstance(value, bytes):
            return value.hex()
        if isinstance(value, (tuple, list)):
            return [encode(x) for x in value]
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        return repr(value)

    functions = [prepare_fast_index, fast_input_block, identical_input_blocks,
                 FastInputData.__init__, FastInputData.block, fast_extension_hash,
                 base_engine_hash, fast_input_runtime.__wrapped__]
    return digest([encode(function.__code__) for function in functions])


def base_engine_hash():
    """Validasi engine asli; identitas loader tambahan dicatat dalam receipt terpisah."""
    active = {name: globals()[name] for name in ['HoldoutData', 'runtime_code_hash', 'run_lock']}
    try:
        globals().update(HoldoutData=ReferenceData, runtime_code_hash=reference_hash, run_lock=reference_lock)
        return reference_hash()
    finally:
        globals().update(active)


@contextmanager
def fast_input_runtime(root, resume_run, enabled=True):
    """Runtime extension tercatat; tidak mengubah kontrak, checkpoint, atau aturan training."""
    if not enabled:
        yield
        return
    require(HoldoutData is ReferenceData and runtime_code_hash is reference_hash,
            'Restart kernel sebelum memasang runtime percepatan')
    original_hash = reference_hash()
    extension = dict(version='holdout_exact_input_loader_v1',
                     original_engine_sha256=original_hash, extension_sha256=fast_extension_hash(),
                     training_updates_unchanged=True, all_groups_and_hours_retained=True,
                     per_group_exact_input_gate=True)
    report = None

    def record(event):
        if report is not None:
            with (report / 'input_acceleration_events.jsonl').open('a', encoding='utf-8') as stream:
                stream.write(json.dumps(dict(utc=datetime.now(timezone.utc).isoformat(),
                                             extension_sha256=extension['extension_sha256'], **event)) + '\n')

    @contextmanager
    def audited_lock(path):
        nonlocal report
        with reference_lock(path):
            report = Path(root) / 'outputs' / Path(path).parent.name
            report.mkdir(parents=True, exist_ok=True)
            contract_path = report / 'model_contract.json'
            if resume_run:
                require(contract_path.is_file(), 'Kontrak run sumber tidak ditemukan')
            if contract_path.exists():
                require(read_json(contract_path)['code_sha256'] == original_hash,
                        'Engine asli berbeda dari checkpoint; percepatan ditolak')
            receipt = report / 'input_acceleration_runtime.json'
            if receipt.exists():
                require(read_json(receipt) == extension, 'Runtime percepatan berbeda')
            else:
                atomic_json(receipt, extension)
            record(dict(event='FAST_INPUT_RUNTIME_STARTED', base_engine_validated=True))
            print('Input loader cepat aktif; engine asli tetap divalidasi; tiap kelompok diuji identik.', flush=True)
            yield

    # Factory hanya mengubah pembacaan input. Model/optimizer/scheduler tetap definisi asli.
    class RuntimeData(FastInputData):
        def __init__(self, inputs):
            super().__init__(inputs)
            self._fast_event = record

    globals().update(HoldoutData=RuntimeData, runtime_code_hash=base_engine_hash, run_lock=audited_lock)
    try:
        require(base_engine_hash() == original_hash, 'Validasi engine dasar gagal')
        yield
    finally:
        globals().update(HoldoutData=ReferenceData, runtime_code_hash=reference_hash, run_lock=reference_lock)

In [70]:
def read_baseline(root, run_id):
    """Baca konfigurasi DGT intermediate 64 yang sudah selesai."""
    source_data = root / "data" / run_id
    source_report = root / "outputs" / run_id
    meta = read_json(source_report / "run_metadata.json")
    contract = read_json(source_report / "model_contract.json")
    fingerprint = digest(contract)
    require(
        meta["run_id"] == run_id and meta["fingerprint"] == fingerprint
        and contract["code_sha256"] == runtime_code_hash()
        and contract["trials"] == trial_definitions()
        and contract["selection"] == "validation_mse_days2"
        and contract["graph_k"] == GRAPH_K == 10,
        "Kontrak berbeda. Gunakan run 70:15:15 dan kernel Step 05 yang sama.",
    )
    choices_file = source_report / "benchmark_choices.json"
    require(
        read_json(source_report / "benchmark_receipt.json")
        == dict(fingerprint=fingerprint, choices_sha256=sha(choices_file)),
        "Catatan benchmark Step 05 tidak cocok.",
    )
    name = "DGT_intermediate_lr_0001"
    trial = next(t for t in contract["trials"] if t["name"] == name)
    execution = read_json(choices_file)[name]
    baseline_dir = source_data / "trials" / name
    baseline_report = source_report / "trials" / name
    baseline = read_json(baseline_dir / "complete.json")
    require(
        baseline["status"] == "COMPLETE"
        and baseline["fingerprint"] == digest(dict(base=fingerprint, trial=trial, execution=execution))
        and baseline["cfg"] == trial["cfg"]
        and baseline["family"] == "DGT" and baseline["fusion"] == "intermediate"
        and trial["cfg"]["d_model"] == 64 and trial["cfg"]["k"] == 10
        and trial["cfg"]["lr"] == 0.0001 and trial["cfg"]["graph_layers"] == 4
        and sha(baseline_dir / "best_LOCAL.pt") == baseline["best_sha256"],
        "Trial DGT intermediate 64 belum selesai atau checkpoint berubah.",
    )
    return meta, contract, trial, execution, baseline, baseline_report

In [71]:
def find_128_run(root, baseline_run, requested_run):
    """Cari run dimensi 128; jangan memilih otomatis jika ada lebih dari satu."""
    if requested_run:
        require(Path(requested_run).name == requested_run
                and "/" not in requested_run and "\\" not in requested_run,
                "Isi DGT128_RUN dengan nama folder run saja.")
        return requested_run
    matches = []
    for path in (root / "outputs").glob(f"{baseline_run}_dgt_inter_d128_*"):
        manifest = path / "experiment.json"
        if not manifest.is_file():
            continue
        trial = read_json(manifest)
        if (trial.get("baseline_run") == baseline_run
                and trial.get("family") == "DGT" and trial.get("fusion") == "intermediate"
                and trial.get("cfg", {}).get("d_model") == 128):
            matches.append(path.name)
    require(len(matches) == 1,
            "Isi DGT128_RUN dari folder hasil training. Run ditemukan: " + repr(sorted(matches)))
    return matches[0]

## Verifikasi sumber dan pemilihan kasus

Hanya test, valid pada dua landmark, pasien unik, seed42. Sampling tidak memakai error
model maupun atribusi. Registry berkode dan pemetaan cohort disimpan sebagai file LOCAL.
Sampel tidak dipilih ulang saat beralih dari pilot ke full. Kuota yang tidak tercapai dilaporkan.

In [72]:
from contextlib import ExitStack, contextmanager
import html
import inspect
import types
from torch.utils.checkpoint import checkpoint as activation_checkpoint


def xai_code_hash():
    """Hash kode notebook, tidak mencakup nilai konfigurasi atau output sel."""
    def encode(code):
        if isinstance(code, types.CodeType):
            return [code.co_code.hex(), [encode(c) for c in code.co_consts],
                    code.co_names, code.co_varnames, code.co_freevars, code.co_cellvars,
                    code.co_flags, code.co_argcount, code.co_kwonlyargcount,
                    code.co_posonlyargcount, code.co_exceptiontable.hex()]
        if isinstance(code, (tuple, list)):
            return [encode(c) for c in code]
        if isinstance(code, bytes):
            return code.hex()
        if code is None or isinstance(code, (str, int, float, bool)):
            return code
        return repr(code)
    functions = []
    for name, value in sorted(globals().items()):
        if callable(value) and (name.startswith('xai_') or name == 'run_step07'):
            if inspect.isfunction(value):
                functions.append((name, encode(inspect.unwrap(value).__code__)))
    return digest(functions)


def xai_read_model(path, completion, cfg, fingerprint):
    require(path.is_file() and sha(path) == completion['best_sha256'],
            'Checkpoint berubah: ' + str(path))
    saved = torch.load(path, map_location='cpu', weights_only=True)
    require(completion['status'] == 'COMPLETE' and completion['fingerprint'] == fingerprint
            and completion['cfg'] == cfg and completion['family'] == 'DGT'
            and completion['fusion'] == 'intermediate' and not completion['test_evaluated']
            and 1 <= completion['best_epoch'] <= completion['epochs'], 'Trial belum valid atau selesai.')
    require(saved['fingerprint'] == fingerprint and saved['cfg'] == cfg
            and saved['family'] == 'DGT' and saved['fusion'] == 'intermediate'
            and saved['selected_epoch'] == completion['best_epoch']
            and saved['best_validation_mse_days2'] == completion['best_validation_mse_days2'],
            'Isi checkpoint tidak sesuai catatan training.')
    return dict(checkpoint=str(path.resolve()), sha256=completion['best_sha256'], cfg=cfg,
                fingerprint=fingerprint, epoch=completion['best_epoch'])


def xai_sources(root, baseline_run, requested128):
    meta, contract, trial, execution, baseline, _ = read_baseline(root, baseline_run)
    base = root / 'data' / baseline_run
    cfg64 = trial['cfg']
    model64 = xai_read_model(base/'trials'/trial['name']/'best_LOCAL.pt', baseline,
                            cfg64, baseline['fingerprint'])
    run128 = find_128_run(root, baseline_run, requested128)
    report128 = root/'outputs'/run128
    experiment = read_json(report128/'experiment.json')
    cfg128 = dict(cfg64, name='lr_0001_d128', d_model=128)
    expected = dict(protocol='holdout70_dgt_intermediate_dimension_ablation_v1',
                    baseline_run=baseline_run, baseline_trial=trial['name'],
                    baseline_fingerprint=baseline['fingerprint'], baseline_sha256=baseline['best_sha256'],
                    family='DGT', fusion='intermediate', cfg=cfg128, settings=contract['settings'],
                    execution=execution, environment=experiment.get('environment'),
                    loader_sha256=fast_extension_hash(), test_scored=False)
    require(experiment == expected, 'Run 128 berbeda dari eksperimen yang ditetapkan.')
    metadata128 = read_json(report128/'run_metadata.json')
    require(metadata128['run_id'] == run128
            and metadata128['status'] == 'TRAINING_COMPLETE_NO_TEST_SCORING'
            and metadata128['fingerprint'] == digest(experiment), 'Training 128 belum selesai.')
    folder128 = root/'data'/run128/'trials'/'DGT_intermediate_lr_0001_d128'
    model128 = xai_read_model(folder128/'best_LOCAL.pt', read_json(folder128/'complete.json'),
                             cfg128, digest(experiment))
    for model in [model64, model128]:
        cfg = model['cfg']
        require(cfg['graph_layers'] == 4 and cfg['k'] == 10 and cfg['lr'] == .0001,
                'Step07 hanya untuk intermediate empat lapisan, k10, LR0.0001.')
    return meta, contract, execution, {'DGT64':model64, 'DGT128':model128}, run128


def xai_sample(ds, settings):
    """Sampling sebelum atribusi; pasien yang telah terpilih tidak diambil kembali."""
    first, second = settings['los_cutoffs_days']
    eligible = np.flatnonzero((ds.role == ROLES['test']) & (ds.lengths >= max(settings['hours'])))
    rng = np.random.default_rng(settings['seed'])
    labels = np.where(ds.y <= first, 'pendek', np.where(ds.y <= second, 'sedang', 'panjang'))
    rows, used_subjects, shortages = [], set(), []
    for label in ['pendek', 'sedang', 'panjang']:
        candidates = rng.permutation(eligible[labels[eligible] == label])
        chosen = []
        for row in candidates:
            subject = int(ds.subjects[row])
            if subject not in used_subjects:
                chosen.append(int(row)); used_subjects.add(subject)
            if len(chosen) == settings['per_group']:
                break
        require(len(chosen) > 0, 'Tidak ada kasus test yang memenuhi syarat pada kelompok ' + label)
        if len(chosen) < settings['per_group']:
            shortages.append(dict(group=label, requested=settings['per_group'], available=len(chosen)))
        # Urutan acak tetap; dua pertama untuk dokter, pertama untuk pilot.
        for i, row in enumerate(chosen):
            rows.append(dict(case_id=f'K{len(rows)+1:03d}', cohort_row=row, los_group=label,
                             clinician_case=i < settings['doctor_per_group'], pilot_case=i == 0))
    return pd.DataFrame(rows), shortages


def xai_schema(ds):
    params = read_json(ds.inputs['numeric']['structured_inputs']['parameters_file'])
    static_groups = {}
    for i, name in enumerate(ds.feature_names):
        if name.startswith('observed:'):
            continue
        key = name.split('=',1)[0]
        static_groups.setdefault(key, []).append(i)
    ref = np.zeros(ds.S, np.float32)
    train = ds.role == ROLES['train']
    groups = []
    for name, indices in static_groups.items():
        if '=' in ds.feature_names[indices[0]]:
            counts = ds.static[train][:,indices].sum(0)
            ref[indices[int(counts.argmax())]] = 1
        groups.append(dict(feature='static:'+name, modality='static', key='static', indices=indices))
    # Observation masks are context, not attribution variables.
    for i,name in enumerate(ds.feature_names):
        if name.startswith('observed:'):
            ref[i] = np.nan  # later replaced with the actual patient's mask
    first_partition = int(ds.seq.partition.iloc[0])
    raw = ds.ledger.loc[first_partition].raw_path
    with np.load(raw, allow_pickle=False) as z:
        names = z['feature_names'].tolist()
    require(len(names) == 43 and len(set(names)) == 43, 'Nama fitur temporal tidak valid.')
    for i,name in enumerate(names):
        groups.append(dict(feature='temporal:'+str(name), modality='temporal', key='numeric', indices=[i]))
    groups.extend([dict(feature='text:notes_embedding', modality='text', key='notes', indices=[]),
                   dict(feature='text:diagnosis_embedding', modality='text', key='diagnosis', indices=[])])
    return dict(groups=groups, static_reference=[float(v) if np.isfinite(v) else None for v in ref], temporal_names=names, parameters=params)


def xai_reference(actual, schema):
    ref = {key: torch.zeros_like(value) for key,value in actual.items()}
    vector = torch.tensor([float('nan') if v is None else v for v in schema['static_reference']], device=actual['static'].device,
                          dtype=actual['static'].dtype)
    ref['static'] = torch.where(torch.isnan(vector), actual['static'], vector)
    return ref


@contextmanager
def xai_gradient_backend():
    """cuDNN eval LSTM tidak menyediakan backward; gunakan backend diferensiabel."""
    fast = torch.backends.mha.get_fastpath_enabled()
    try:
        torch.backends.mha.set_fastpath_enabled(False)
        with torch.backends.cudnn.flags(enabled=False):
            from torch.nn.attention import sdpa_kernel, SDPBackend
            with sdpa_kernel(SDPBackend.MATH):
                yield
    finally:
        torch.backends.mha.set_fastpath_enabled(fast)

## Integrated Gradients dan fidelity

Implementasi Gauss Legendre setara definisi IG. Hitung ulang pada 32/64/128 langkah
sampai completeness dan perubahan atribusi memenuhi toleransi. Penjelasan yang gagal
tidak dimasukkan ke global atau paket dokter. Fidelity memakai penggantian kelompok
fitur dengan referensi dan kontrol acak berukuran sama; tidak harus menurunkan prediksi.

Analisis graf menghapus hubungan incoming pasien utama dengan satu tetangga sepanjang
prefix. Kandidat dipilih menurut prior kemiripan kumulatif, bukan bobot attention.
Efek yang dilaporkan adalah perubahan prediksi, dengan self connection tetap ada.

In [73]:
XAI_INPUT_KEYS = ('static', 'numeric', 'notes', 'diagnosis')
XAI_TIME_KEYS = ('numeric', 'numeric_mask', 'notes', 'notes_mask', 'diagnosis_mask')


def xai_context(ds, rows, local_index, cache, hour, device):
    batch, counts = ds.block(rows, 0, hour)
    graphs = graph_block(cache, 'DGT', 0, hour, 10, device)
    return dict(batch=tensor_batch(batch, device), graphs={'balanced':graphs['balanced']},
                local_index=local_index, counts=counts, rows=rows)


def xai_actual(context, hour):
    i = context['local_index']
    return {key:(context['batch'][key][i,:hour] if key in ('numeric','notes')
                 else context['batch'][key][i]).detach().clone() for key in XAI_INPUT_KEYS}


def xai_forward(model, context, values, hour, block, checkpoint_blocks=False, graphs=None):
    """State dihitung dari jam 1, tanpa detach. Tetangga dan mask tetap aktual."""
    base = context['batch']
    index = torch.tensor([context['local_index']], device=base['static'].device)
    merged = dict(base)
    for key in XAI_INPUT_KEYS:
        original = base[key][:,:hour] if key in ('numeric','notes') else base[key]
        merged[key] = original.index_copy(0, index, values[key].unsqueeze(0))
    edges = graphs if graphs is not None else context['graphs']
    state = None
    for start in range(0, hour, block):
        stop = min(start+block,hour)
        b = {key:(value[:,start:stop] if key in XAI_TIME_KEYS else
                  value[start:stop] if key=='hours' else value)
             for key,value in merged.items()}
        g = {view:(e[start:stop],w[start:stop]) for view,(e,w) in edges.items()}
        if checkpoint_blocks and torch.is_grad_enabled():
            output,state = activation_checkpoint(model,b,g,state,use_reentrant=False,
                                                 preserve_rng_state=False)
        else:
            output,state = model(b,g,state)
    return output[context['local_index'],-1]


def xai_ig(model, context, actual, reference, hour, block, settings):
    """IG Gauss Legendre, satu titik integrasi per forward untuk membatasi VRAM."""
    with torch.no_grad():
        prediction = float(xai_forward(model,context,actual,hour,block))
        base_prediction = float(xai_forward(model,context,reference,hour,block))
    difference = prediction-base_prediction
    previous = None
    steps = settings['ig_steps']
    attempts = []
    while steps <= settings['ig_max_steps']:
        nodes,weights = np.polynomial.legendre.leggauss(steps)
        accum = {key:torch.zeros_like(actual[key],dtype=torch.float64) for key in XAI_INPUT_KEYS}
        for alpha,weight in zip((nodes+1)/2,weights/2):
            point = {key:(reference[key]+float(alpha)*(actual[key]-reference[key]))
                     .detach().requires_grad_(True) for key in XAI_INPUT_KEYS}
            output = xai_forward(model,context,point,hour,block,settings['checkpoint_blocks'])
            gradients = torch.autograd.grad(output,tuple(point.values()),allow_unused=True)
            for key,gradient in zip(point,gradients):
                if gradient is not None:
                    accum[key].add_(gradient.detach().to(torch.float64),alpha=float(weight))
            del output,gradients,point
        attrs = {key:((actual[key]-reference[key]).to(torch.float64)*accum[key]).cpu().numpy()
                 for key in XAI_INPUT_KEYS}
        require(all(np.isfinite(a).all() for a in attrs.values()),'Atribusi tidak finite.')
        total = sum(float(a.sum()) for a in attrs.values())
        error = abs(total-difference)
        stability = None
        if previous is not None:
            numerator = sum(float(np.abs(attrs[k]-previous[k]).sum()) for k in attrs)
            denominator = max(sum(float(np.abs(attrs[k]).sum()) for k in attrs),1e-12)
            stability = numerator/denominator
        passed = (error <= settings['ig_abs_tolerance_days']+settings['ig_rel_tolerance']*abs(difference)
                  and stability is not None and stability <= settings['ig_stability_tolerance'])
        attempts.append(dict(steps=steps,completeness_error_days=error,relative_l1_change=stability,passed=bool(passed)))
        print(f'    IG {steps}: error={error:.6g} hari; stabilitas={stability}',flush=True)
        if passed or steps==settings['ig_max_steps']:
            return attrs,dict(prediction_days=prediction,reference_prediction_days=base_prediction,
                              attribution_sum_days=total,numerical_pass=bool(passed),attempts=attempts)
        previous=attrs
        steps=min(steps*2,settings['ig_max_steps'])
    raise ValueError('Rentang langkah IG tidak valid.')


def xai_feature_table(attrs,schema):
    records=[]
    for group in schema['groups']:
        array=attrs[group['key']]
        if group['indices']:
            array=array[...,group['indices']]
        value=float(array.sum())
        records.append(dict(feature=group['feature'],modality=group['modality'],
                            attribution_days=value,importance=abs(value)))
    return pd.DataFrame(records)


def xai_replace_groups(actual,reference,schema,indices):
    result={key:value.clone() for key,value in actual.items()}
    for i in indices:
        group=schema['groups'][int(i)];key=group['key'];columns=group['indices']
        if columns:
            result[key][...,columns]=reference[key][...,columns]
        else:
            result[key]=reference[key].clone()
    return result


@torch.no_grad()
def xai_fidelity(model,context,actual,reference,schema,features,hour,block,settings):
    before=float(xai_forward(model,context,actual,hour,block))
    ranking=np.argsort(-features.importance.to_numpy(),kind='stable')
    rng=np.random.default_rng(settings['seed']+int(context['rows'][context['local_index']])*100+hour)
    records=[]
    for count in settings['fidelity_counts']:
        count=min(int(count),len(ranking));top=ranking[:count]
        changed=xai_replace_groups(actual,reference,schema,top)
        after=float(xai_forward(model,context,changed,hour,block))
        random_changes=[]
        for repeat in range(settings['random_repeats']):
            chosen=rng.choice(len(ranking),count,replace=False)
            x=xai_replace_groups(actual,reference,schema,chosen)
            random_changes.append(float(xai_forward(model,context,x,hour,block))-before)
        records.append(dict(features_removed=count,features='; '.join(features.feature.iloc[top]),
                            prediction_before_days=before,prediction_after_days=after,
                            signed_change_days=after-before,absolute_change_days=abs(after-before),
                            random_absolute_mean_days=float(np.mean(np.abs(random_changes))),
                            random_signed_changes_days=random_changes))
    return records


@torch.no_grad()
def xai_graph_effect(model,context,actual,hour,block,settings):
    i=context['local_index'];edge,weight=context['graphs']['balanced']
    e=edge[:hour].cpu().numpy();w=weight[:hour].cpu().numpy()
    scores={}
    for neighbour,prior in zip(e[:,i].ravel(),w[:,i].ravel()):
        if neighbour>=0:scores[int(neighbour)]=scores.get(int(neighbour),0.)+float(prior)
    candidates=sorted(scores,key=lambda n:(-scores[n],n))[:settings['graph_neighbours_to_probe']]
    before=float(xai_forward(model,context,actual,hour,block))
    records=[]
    for rank,neighbour in enumerate(candidates,1):
        changed_e=edge.clone();changed_w=weight.clone()
        remove=changed_e[:,i].eq(neighbour)
        changed_e[:,i]=changed_e[:,i].masked_fill(remove,-1)
        changed_w[:,i]=changed_w[:,i].masked_fill(remove,0)
        after=float(xai_forward(model,context,actual,hour,block,
                                graphs={'balanced':(changed_e,changed_w)}))
        records.append(dict(neighbour_code=f'N{rank:02d}',local_neighbour=neighbour,
                            selection='largest_cumulative_similarity_prior',
                            edge_hours_removed=int(remove.any(1).sum()),
                            signed_change_days=after-before,absolute_change_days=abs(after-before)))
    return records

## Ekspor hasil dan paket penilaian dokter

`global_features.csv`, `global_by_los.csv`, `global_modalities.csv`, `model_comparison.csv`,
`fidelity.csv`, dan grafik global merangkum sampel yang lolos pemeriksaan numerik.
Perbandingan model memakai kasus berpasangan yang sama-sama lolos.

Folder `doctor_packets` menyediakan index HTML, form offline per kasus dan landmark,
serta template CSV. Dokter dapat mengunduh jawaban JSON sendiri; tidak ada pengiriman
atau pembuatan Google Form otomatis. Model diberi label A/B; kunci disimpan terpisah
sebagai `blinding_key_LOCAL.csv`. LOS aktual dan kelompok LOS tidak dimuat di HTML dokter.

**Sebelum diberikan kepada dokter:** periksa label dan satuan setiap fitur menggunakan
kamus ekstraksi sumber. File `feature_dictionary_template.csv` disediakan untuk pencatatan;
edit file tersebut tidak otomatis mengubah grafik. Skor form merupakan draft, bukan
instrumen yang sudah tervalidasi. Pilot satu kasus bersama pembimbing/pakar lebih dulu.
Jam48 ditinjau lebih dahulu; penilaian jam24 selanjutnya dapat dipengaruhi informasi48.
Form yang membutuhkan isi catatan diberi pilihan tidak dapat menilai karena teks mentah
belum disediakan. Simpan file LOCAL sebagai catatan peneliti, terpisah dari paket dokter.

In [74]:
import base64
import matplotlib.pyplot as plt


def xai_save_arrays(path,attrs):
    temporary=path.with_suffix('.tmp')
    with temporary.open('wb') as stream:
        np.savez_compressed(stream,**attrs)
    os.replace(temporary,path)


def xai_plot_local(table,path,title,top=15):
    chosen=table.nlargest(min(top,len(table)),'importance').sort_values('importance')
    fig,ax=plt.subplots(figsize=(9,6))
    ax.barh(chosen.feature,chosen.attribution_days,
            color=['#b34d37' if x>=0 else '#287b93' for x in chosen.attribution_days])
    ax.axvline(0,color='black',lw=.7)
    ax.set(title=title,xlabel='Kontribusi terhadap prediksi total LOS (hari), relatif terhadap referensi')
    fig.tight_layout();fig.savefig(path,dpi=150);plt.close(fig)


def xai_plot_heatmap(attrs,names,path):
    values=attrs['numeric'].T
    bound=max(float(np.abs(values).max()),1e-8)
    fig,ax=plt.subplots(figsize=(12,10))
    plot=ax.imshow(values,aspect='auto',cmap='RdBu_r',vmin=-bound,vmax=bound,
                   extent=(.5,values.shape[1]+.5,len(names)-.5,-.5))
    ax.set_yticks(range(len(names)),names,fontsize=8)
    ax.set(xlabel='Jam sejak masuk ICU',title='Atribusi fitur temporal sampai landmark')
    fig.colorbar(plot,ax=ax,label='Kontribusi pada LOS (hari)')
    fig.tight_layout();fig.savefig(path,dpi=130);plt.close(fig)


def xai_clinical_context(ds,context,actual,schema,hour):
    x=actual['numeric'].cpu().numpy()
    mask=context['batch']['numeric_mask'][context['local_index'],:hour].cpu().numpy()
    params=schema['parameters'];p=params['temporal']
    raw=x*np.asarray(p['scale'])+np.asarray(p['mean'])
    rows=[]
    for i,name in enumerate(schema['temporal_names']):
        rows.append(dict(feature='temporal:'+str(name),last_value=float(raw[-1,i]),
                         mean_value=float(raw[:,i].mean()),min_value=float(raw[:,i].min()),
                         max_value=float(raw[:,i].max()),observed_hours=int(mask[:,i].sum()),
                         total_hours=hour,last_observed=bool(mask[-1,i]),
                         source_note='Nilai input model; jam tanpa observasi dapat berisi imputasi atau forward fill.'))
    static=actual['static'].cpu().numpy();profile=[]
    for group in schema['groups']:
        if group['key']!='static':continue
        indices=group['indices'];name=group['feature']
        if '=' in ds.feature_names[indices[0]]:
            selected=indices[int(static[indices].argmax())]
            value=ds.feature_names[selected].split('=',1)[1]
        elif name=='static:age_z':
            value=f"{static[indices[0]]*params['static']['age_scale']+params['static']['age_mean']:.1f} tahun"
        else:value=f'{static[indices[0]]:.3f} (input model)'
        profile.append(dict(feature=name,value=value))
    return rows,profile


def xai_doctor_packet(case,hour,results,report,settings):
    """HTML mandiri, tanpa target aktual, ID sumber, kelompok LOS atau nama dimensi."""
    folder=report/'doctor_packets'/f'hour_{hour}'
    folder.mkdir(parents=True,exist_ok=True)
    rng=np.random.default_rng(settings['seed']+int(case.case_id[1:]))
    order=list(rng.permutation(['DGT64','DGT128']))
    mapping=[dict(case_id=case.case_id,hour=hour,label=label,model=model)
             for label,model in zip(['A','B'],order)]
    esc=lambda v:html.escape(str(v))
    def picture(path):
        return '<img src="data:image/png;base64,'+base64.b64encode(path.read_bytes()).decode()+'">'
    first=results[order[0]]
    profile=''.join('<tr><td>'+esc(v['feature'])+'</td><td>'+esc(v['value'])+'</td></tr>' for v in first['profile'])
    clinical=''.join('<tr>'+''.join('<td>'+esc(v)+'</td>' for v in [r['feature'],f"{r['last_value']:.3g}",
                      f"{r['mean_value']:.3g}",f"{r['min_value']:.3g}–{r['max_value']:.3g}",
                      f"{r['observed_hours']}/{hour}",'Ya' if r['last_observed'] else 'Tidak'])+'</tr>'
                     for r in first['clinical'])
    questions=['Nama fitur mudah dipahami.','Grafik kontribusi mudah dibaca.',
               'Fitur dominan relevan terhadap kondisi kasus.','Arah kontribusi masuk akal dalam konteks kasus.',
               'Penjelasan sesuai dengan informasi sampai landmark.','Penjelasan secara keseluruhan masuk akal secara klinis.',
               'Nilai dan tren yang ditampilkan cukup untuk menilai penjelasan.',
               'Penjelasan membantu memahami alasan prediksi model.']
    def select(name):
        return '<select name="'+name+'"><option value="">Pilih</option>'+''.join(
            f'<option value="{x}">{x}</option>' for x in ['1','2','3','4','Tidak dapat menilai'])+'</select>'
    parts=[]
    for label,model in zip(['A','B'],order):
        result=results[model]
        if not result['quality']['numerical_pass']:
            parts.append('<h2>Model '+label+'</h2><p>Atribusi belum lolos pemeriksaan numerik. Penilaian ditunda.</p>')
            continue
        path=report/model/case.case_id/f'hour_{hour}'
        attrs=pd.read_csv(path/'feature_attributions.csv').nlargest(5,'importance')
        rating=''.join('<tr><td>'+str(i)+'</td><td>'+q+'</td><td>'+select(label+'_P'+str(i))+'</td></tr>'
                       for i,q in enumerate(questions,1))
        features=''.join('<tr><td>'+esc(r.feature)+'</td><td>'+f'{r.attribution_days:.4f}'+'</td><td><input name="'+
                         label+'_feature_'+str(i)+'" placeholder="Relevansi, arah, dan alasan"></td></tr>'
                         for i,r in enumerate(attrs.itertuples(),1))
        graph_rows=''.join('<tr><td>'+esc(r['neighbour_code'])+'</td><td>'+f"{r['signed_change_days']:.4f}"+
                           '</td></tr>' for r in result['graph_effect'])
        neighbour_profile=''.join('<tr><td>'+esc(r['neighbour_code'])+'</td><td>'+esc(r['profile'])+'</td></tr>'
                                   for r in result.get('neighbour_profiles',[]))
        parts.append('<details><summary>Model '+label+' — buka setelah telaah klinis awal</summary><h2>Model '+label+
                     '</h2><p>Prediksi total LOS: '+f"{result['quality']['prediction_days']:.3f}"+' hari.</p>'+
                     picture(path/'local_top15.png')+picture(path/'temporal_heatmap.png')+
                     '<h3>Penilaian penjelasan</h3><table><tr><th>No</th><th>Pernyataan</th><th>Skor</th></tr>'+rating+
                     '</table><h3>Telaah lima fitur dominan</h3><table><tr><th>Fitur</th><th>Kontribusi hari</th><th>Penilaian</th></tr>'+
                     features+'</table><h3>Konteks graf</h3><table><tr><th>Tetangga</th><th>Ringkasan input sampai landmark</th></tr>'+neighbour_profile+
                     '</table><table><tr><th>Hubungan yang dihilangkan</th><th>Perubahan prediksi hari</th></tr>'+graph_rows+
                     '</table><p>Hubungan masuk akal? <input name="'+label+'_graph_review"></p>'+
                     '<p>Komentar dan perbaikan:<br><textarea name="'+label+'_comment"></textarea></p></details>')
    intro=f'''<!doctype html><html lang="id"><meta charset="utf-8"><title>Kasus {case.case_id} jam {hour}</title>
<style>body{{font:16px Arial,sans-serif;max-width:1050px;margin:35px auto;padding:20px;color:#222;line-height:1.5}}
table{{border-collapse:collapse;width:100%;margin:15px 0}}th,td{{border:1px solid #ccc;padding:8px;text-align:left}}
th{{background:#edf2f5}}img{{max-width:100%}}textarea{{width:98%;height:80px}}input{{max-width:95%;padding:5px}}
summary{{font-weight:bold;cursor:pointer;padding:14px;background:#edf2f5}}details{{margin:20px 0}}
@media print{{button{{display:none}}}}</style><h1>Penilaian penjelasan prediksi ICU LOS</h1>
<p>Kasus {case.case_id} | Landmark jam ke-{hour}. Informasi dibatasi sampai landmark.</p>
<p>Skala: 1 sangat tidak setuju; 2 tidak setuju; 3 setuju; 4 sangat setuju. Pilih tidak dapat menilai jika informasi tidak cukup.</p>
<p>Atribusi positif menaikkan prediksi relatif terhadap referensi; negatif menurunkannya. Penjelasan bukan bukti sebab akibat atau rekomendasi terapi.</p>
<form id="review"><p>Kode pakar: <input name="expert_code"></p>
<p>Bidang kompetensi: <input name="specialty"> Pengalaman ICU (tahun): <input name="icu_years" type="number" min="0"></p>
<p>Kecukupan informasi kasus: <input name="case_information_sufficiency" placeholder="Cukup / sebagian / tidak cukup"></p><h2>Telaah klinis awal</h2>
<p>Catat sampai tiga faktor yang relevan sebelum membuka penjelasan model:</p><textarea name="clinical_factors"></textarea>
<p>Informasi yang belum tersedia:</p><textarea name="missing_information"></textarea>
<table><tr><th>Karakteristik</th><th>Nilai</th></tr>{profile}</table>
<p>Variabel temporal ditampilkan pada satuan sumber masing-masing. Jangan membandingkan besaran antarvariabel sebagai satuan yang sama.</p>
<table><tr><th>Fitur</th><th>Terakhir</th><th>Rerata</th><th>Min–maks</th><th>Jam terukur</th><th>Terukur di jam terakhir</th></tr>{clinical}</table>
<p>Nilai tanpa observasi dapat merupakan imputasi atau forward fill. Mask observasi dipertahankan pada atribusi.
Konten catatan dan sorotan kata tidak diekspor; kontribusi teks hanya pada tingkat embedding. Item yang memerlukan isi teks dapat dinilai tidak dapat menilai.</p>'''
    end='''<h2>Perbandingan penjelasan</h2><p>Mana lebih mudah dipahami? <input name="comparison_understanding" placeholder="A / B / setara / tidak dapat menilai"></p>
<p>Mana lebih masuk akal secara klinis? <input name="comparison_plausibility"></p>
<p>Mana lebih membantu memahami prediksi? <input name="comparison_usefulness"></p>
<p>Alasan:<br><textarea name="comparison_reason"></textarea></p></form>
<button type="button" id="save">Unduh jawaban JSON</button><p>Jawaban hanya disimpan ketika tombol unduh ditekan. Tidak ada pengiriman otomatis.</p>
<script>document.getElementById('save').onclick=()=>{const data=Object.fromEntries(new FormData(document.getElementById('review')));
DATA_METADATA;const a=document.createElement('a');a.href=URL.createObjectURL(new Blob([JSON.stringify(data,null,2)],{type:'application/json'}));
a.download=NAME_JSON;a.click();setTimeout(()=>URL.revokeObjectURL(a.href),1000);};</script></html>'''
    end=end.replace('DATA_METADATA','data.case_id='+json.dumps(case.case_id)+';data.hour='+str(hour)+';')
    end=end.replace('NAME_JSON',json.dumps(f'review_{case.case_id}_{hour}h.json'))
    (folder/(case.case_id+'.html')).write_text(intro+''.join(parts)+end,encoding='utf-8')
    return mapping


def xai_aggregate(report,sample,records,schema,settings,mode):
    summary=[];features=[];fidelity=[];graph=[];mapping=[]
    by_key={(r['model'],r['case_id'],r['hour']):r for r in records}
    for result in records:
        base={key:result[key] for key in ['model','case_id','hour','los_group']}
        quality=result['quality']
        summary.append(base|{k:quality[k] for k in ['prediction_days','reference_prediction_days','numerical_pass']}|
                       dict(selected_epoch=result['selected_epoch'],total_los_days=result['total_los_days'],
                            ig_steps=quality['attempts'][-1]['steps'],
                            completeness_error_days=quality['attempts'][-1]['completeness_error_days'],
                            prediction_agreement_max_abs=result['prediction_agreement_max_abs']))
        if quality['numerical_pass']:
            frame=pd.read_csv(report/result['model']/result['case_id']/f"hour_{result['hour']}"/'feature_attributions.csv')
            for row in frame.to_dict('records'):features.append(base|row)
            for row in result['fidelity']:fidelity.append(base|row)
            for row in result['graph_effect']:graph.append(base|row)
    pd.DataFrame(summary).to_csv(report/'case_results_LOCAL.csv',index=False)
    pd.DataFrame(fidelity).to_csv(report/'fidelity.csv',index=False)
    pd.DataFrame(graph).to_csv(report/'graph_sensitivity_LOCAL.csv',index=False)
    feature_frame=pd.DataFrame(features)
    if not feature_frame.empty:
        feature_frame.to_csv(report/'all_feature_attributions.csv',index=False)
        global_table=feature_frame.groupby(['model','hour','feature','modality'],as_index=False).agg(
            mean_signed_days=('attribution_days','mean'),mean_absolute_days=('importance','mean'),
            cases=('case_id','nunique'))
        global_table['sample_scope']='balanced_LOS_sample_'+mode
        global_table.to_csv(report/'global_features.csv',index=False)
        feature_frame.groupby(['model','hour','los_group','feature'],as_index=False).agg(
            mean_absolute_days=('importance','mean'),cases=('case_id','nunique')).to_csv(report/'global_by_los.csv',index=False)
        modalities=feature_frame.groupby(['model','case_id','hour','modality'],as_index=False).attribution_days.sum()
        modalities['absolute_net_days']=modalities.attribution_days.abs()
        modalities.groupby(['model','hour','modality'],as_index=False).agg(
            mean_signed_days=('attribution_days','mean'),mean_absolute_net_days=('absolute_net_days','mean')).to_csv(
                report/'global_modalities.csv',index=False)
        for (model,hour),frame in global_table.groupby(['model','hour']):
            chosen=frame.nlargest(15,'mean_absolute_days').sort_values('mean_absolute_days')
            fig,ax=plt.subplots(figsize=(9,6));ax.barh(chosen.feature,chosen.mean_absolute_days,color='#287b93')
            ax.set(title=f'{model} | jam {hour} | sampel {mode}',xlabel='Rerata absolut kontribusi bersih per fitur (hari)')
            fig.tight_layout();fig.savefig(report/f'global_{model}_{hour}h.png',dpi=150);plt.close(fig)
        comparisons=[]
        for hour in settings['hours']:
            a=global_table[(global_table.model=='DGT64') & (global_table.hour==hour)].set_index('feature')
            b=global_table[(global_table.model=='DGT128') & (global_table.hour==hour)].set_index('feature')
            if len(a) and len(b):
                # Rankings use paired numerical-PASS cases only in the paired table below.
                paired_ids=set(feature_frame[(feature_frame.model=='DGT64') & (feature_frame.hour==hour)].case_id)&set(
                    feature_frame[(feature_frame.model=='DGT128') & (feature_frame.hour==hour)].case_id)
                pair=feature_frame[feature_frame.hour.eq(hour)&feature_frame.case_id.isin(paired_ids)]
                if pair.empty:continue
                pivot=pair.groupby(['feature','model']).importance.mean().unstack().dropna()
                ra=pivot.DGT64.rank(method='average');rb=pivot.DGT128.rank(method='average')
                corr=float(ra.corr(rb)) if ra.nunique()>1 and rb.nunique()>1 else None
                topa=set(pivot.DGT64.nlargest(15).index);topb=set(pivot.DGT128.nlargest(15).index)
                comparisons.append(dict(hour=hour,paired_cases=len(paired_ids),top15_overlap=len(topa&topb),
                                        top15_jaccard=len(topa&topb)/max(len(topa|topb),1),spearman=corr))
                pivot.to_csv(report/f'paired_global_comparison_{hour}h.csv')
        pd.DataFrame(comparisons).to_csv(report/'model_comparison.csv',index=False)
    if not feature_frame.empty:
        local_comparisons=[]
        for case_id in sample.case_id:
            for hour in settings['hours']:
                a=feature_frame[(feature_frame.case_id==case_id)&(feature_frame.hour==hour)&(feature_frame.model=='DGT64')].set_index('feature')
                b=feature_frame[(feature_frame.case_id==case_id)&(feature_frame.hour==hour)&(feature_frame.model=='DGT128')].set_index('feature')
                if a.empty or b.empty:continue
                keys=a.index.intersection(b.index)
                important=(a.loc[keys,'importance']>1e-8)&(b.loc[keys,'importance']>1e-8)
                signs=np.sign(a.loc[keys,'attribution_days'])==np.sign(b.loc[keys,'attribution_days'])
                topa=set(a.importance.nlargest(15).index);topb=set(b.importance.nlargest(15).index)
                local_comparisons.append(dict(case_id=case_id,hour=hour,top15_overlap=len(topa&topb),
                    features_nonzero_both=int(important.sum()),
                    sign_agreement=None if not important.any() else float(signs[important].mean())))
        pd.DataFrame(local_comparisons).to_csv(report/'local_model_comparison.csv',index=False)
        landmark_comparisons=[]
        for model in ['DGT64','DGT128']:
            for case_id in sample.case_id:
                a=feature_frame[(feature_frame.case_id==case_id)&(feature_frame.hour==settings['hours'][0])&(feature_frame.model==model)].set_index('feature')
                b=feature_frame[(feature_frame.case_id==case_id)&(feature_frame.hour==settings['hours'][1])&(feature_frame.model==model)].set_index('feature')
                if a.empty or b.empty:continue
                topa=set(a.importance.nlargest(15).index);topb=set(b.importance.nlargest(15).index)
                first=by_key[(model,case_id,settings['hours'][0])]['quality']['prediction_days']
                second=by_key[(model,case_id,settings['hours'][1])]['quality']['prediction_days']
                landmark_comparisons.append(dict(model=model,case_id=case_id,primary_hour=settings['hours'][0],
                    comparison_hour=settings['hours'][1],top15_overlap=len(topa&topb),
                    primary_prediction_days=first,comparison_prediction_days=second,
                    prediction_change_days=first-second))
        pd.DataFrame(landmark_comparisons).to_csv(report/'landmark_comparison.csv',index=False)
    packet_links=[]
    for case in sample[sample.clinician_case].itertuples():
        for hour in settings['hours']:
            pair={model:by_key.get((model,case.case_id,hour)) for model in ['DGT64','DGT128']}
            if all(pair.values()) and all(r['quality']['numerical_pass'] for r in pair.values()):
                mapping.extend(xai_doctor_packet(case,hour,pair,report,settings))
                packet_links.append(f'<li><a href="hour_{hour}/{case.case_id}.html">{case.case_id} | jam {hour}</a></li>')
    pd.DataFrame(mapping).to_csv(report/'blinding_key_LOCAL.csv',index=False)
    doctor=report/'doctor_packets';doctor.mkdir(exist_ok=True)
    (doctor/'index.html').write_text('<!doctype html><meta charset="utf-8"><h1>Paket penilaian dokter</h1>'+
        '<p>Utamakan jam 48, lalu jam 24. Hanya paket yang lolos pemeriksaan numerik ditampilkan.</p><ul>'+
        ''.join(packet_links)+'</ul>',encoding='utf-8')
    review_rows=[{**r,**{f'P{i}':'' for i in range(1,9)},'expert_code':'','comment':''} for r in mapping]
    # Mapping model tidak disertakan pada template untuk penilai.
    pd.DataFrame([{k:v for k,v in r.items() if k!='model'} for r in review_rows]).to_csv(
        doctor/'review_template.csv',index=False)
    (doctor/'comparison_24_48.csv').write_text('case_id,expert_code,model_label,change_plausible,comment\n'+
        ''.join(f'{case.case_id},,,,\n' for case in sample[sample.clinician_case].itertuples()),encoding='utf-8')
    (doctor/'instructions.md').write_text(
        '# Petunjuk penilaian\n\n'
        'Jam utama: '+str(settings['hours'][0])+'. Pembanding: '+str(settings['hours'][1])+'.\n'
        'Telaah kondisi kasus sebelum membuka Model A/B. Urutan A/B sama pada dua landmark untuk satu kasus.\n'
        'Skala P1-P8: 1 sangat tidak setuju; 2 tidak setuju; 3 setuju; 4 sangat setuju; atau tidak dapat menilai.\n'
        'Jawaban HTML dapat diunduh sebagai JSON dan diserahkan melalui prosedur penelitian yang disepakati.\n'
        'Penilaian landmark lebih awal setelah melihat landmark utama dapat dipengaruhi pengetahuan informasi berikutnya.\n'
        'Form ini draft untuk pilot pakar, bukan instrumen baku tervalidasi.\n'
        'Periksa satuan sumber fitur sebelum menginterpretasikan nilai. Skor dokter dilaporkan per butir; bukan satu skor validitas total.\n'
        'Teks mentah tidak disajikan; nilai item yang memerlukan konten teks sebagai tidak dapat menilai.\n'
        'Kategori analisis usulan: pendek <= '+str(settings['los_cutoffs_days'][0])+' hari; sedang sampai '+
        str(settings['los_cutoffs_days'][1])+' hari; panjang di atasnya. Pakar dapat memberi usulan revisi.\n',encoding='utf-8')
    return pd.DataFrame(summary)

In [75]:
def run_step07(root,baseline_run,run128,mode,settings):
    require(mode in ('pilot','full'),'MODE harus pilot atau full.')
    require(settings['per_group']>=1 and 1<=settings['doctor_per_group']<=settings['per_group'],
            'Jumlah kasus dokter harus antara satu dan jumlah sampel per kelompok.')
    require(len(settings['hours'])==2 and settings['hours'][0]>settings['hours'][1]>=4,
            'Landmark utama harus lebih akhir; gunakan (48,24).')
    require(2<=settings['ig_steps'] and settings['ig_max_steps']>=2*settings['ig_steps'],
            'Maksimum langkah IG minimal dua kali langkah awal untuk uji stabilitas.')
    require(settings['random_repeats']>=1 and all(c>=1 for c in settings['fidelity_counts']),
            'Kontrol acak dan jumlah fitur fidelity tidak valid.')
    root=Path(root)
    require((root/'data'/baseline_run/'run.lock').is_file(),'Run baseline belum ditemukan.')
    with run_lock(root/'data'/baseline_run/'run.lock'):
        meta,contract,execution,models,source128=xai_sources(root,baseline_run,run128)
        device=execution['device']
        require(torch.device(device).type=='cpu' or (torch.cuda.is_available()
                and (torch.device(device).index or 0)<torch.cuda.device_count()),
                'GPU training tidak tersedia. Pilih kernel CUDA yang dipakai pada training.')
        torch.set_num_threads(execution['threads'])
        manifest=dict(protocol='step07_paired_DGT_IG_v1',baseline_run=baseline_run,run128=source128,
                      models=models,source_fingerprint=meta['fingerprint'],settings=settings,
                      xai_code_sha256=xai_code_hash(),
                      baseline='root numeric z0; train modal onehot; text0; masks and neighbour inputs fixed',
                      graph='actual hourly topology fixed for IG; incoming-root edge deletion separately',
                      output='total LOS days at landmark; full prefix gradient; no state detach',
                      environment=dict(python=platform.python_version(),numpy=np.__version__,
                                       torch=str(torch.__version__),cuda=torch.version.cuda,device=device))
        report=root/'outputs'/'07_xai_70_15_15'/digest(manifest)[:16]
        report.mkdir(parents=True,exist_ok=True)
        with run_lock(report/'run.lock'):
            lock=report/'protocol.json'
            if lock.exists():require(read_json(lock)==manifest,'Protokol laporan berbeda.')
            else:atomic_json(lock,manifest)
            verification=report/'source_verification.json'
            previous=root/'data'/baseline_run/verification.name
            if not verification.exists() and previous.is_file():shutil.copyfile(previous,verification)
            inputs=verify_sources(root,meta['stage03_run'],meta['stage04_run'],report)
            require(inputs['cohort_sha256']==contract['cohort_sha256'] and
                    {p:a['sha256'] for p,a in inputs['watch'].items()}==contract['source_hashes'],
                    'Input berbeda dari training.')
            ds=FastInputData(inputs)
            try:
                sample,shortages=xai_sample(ds,settings)
                selection={'cases':sample.to_dict('records'),'shortages':shortages,
                           'eligibility':'test and valid at both landmarks; unique patients',
                           'seed':settings['seed']}
                sample_lock=report/'locked_sample_LOCAL.json'
                if sample_lock.exists():require(read_json(sample_lock)==selection,'Daftar kasus berubah.')
                else:atomic_json(sample_lock,selection)
                sample.to_csv(report/'sample_registry_LOCAL.csv',index=False)
                schema=xai_schema(ds)
                atomic_json(report/'feature_schema.json',schema)
                pd.DataFrame([{'feature':g['feature'],'clinical_label':g['feature'],'unit':''}
                              for g in schema['groups']]).to_csv(report/'feature_dictionary_template.csv',index=False)
                groups_raw=read_json(root/'data'/baseline_run/'graph_groups_LOCAL.json')
                require(digest(groups_raw)==contract['group_membership_sha256'] and
                        np.array_equal(np.sort(np.concatenate(groups_raw)),np.arange(ds.n)),
                        'Grup graf tidak cocok.')
                groups=[np.asarray(g,dtype=np.int64) for g in groups_raw]
                lookup={int(row):(i,j) for i,g in enumerate(groups) for j,row in enumerate(g)}
                cache_root=root/'data'/baseline_run/'graph_cache'
                active=sample[sample.pilot_case] if mode=='pilot' else sample
                records=[]
                print(f'Mode {mode}: {len(active)} kasus x dua model x dua landmark. Laporan: {report}',flush=True)
                if shortages:print('Jumlah tersedia lebih sedikit dari permintaan:',shortages,flush=True)
                for tag,model_info in models.items():
                    require(sha(model_info['checkpoint'])==model_info['sha256'],'Checkpoint berubah.')
                    saved=torch.load(model_info['checkpoint'],map_location='cpu',weights_only=True)
                    require(saved['static_dim']==ds.S and saved['attention_window']==contract['settings']['attention_window'],
                            'Dimensi input model tidak cocok.')
                    model=LOSModel(ds.S,'DGT','intermediate',model_info['cfg'],
                                   contract['settings']['attention_window']).to(device).eval()
                    model.load_state_dict(saved['model'],strict=True)
                    for parameter in model.parameters():parameter.requires_grad_(False)
                    weights_before={k:v._version for k,v in model.state_dict(keep_vars=True).items()}
                    for case in active.itertuples():
                        group_index,local_index=lookup[int(case.cohort_row)];rows=groups[group_index]
                        folder=cache_root/f'group_{group_index:05d}'
                        require((folder/'commit.json').is_file(),'Cache graf belum tersedia: '+str(folder))
                        cache=graph_cache(ds,rows,folder,10,contract['settings']['block_hours'])
                        for hour in settings['hours']:
                            destination=report/tag/case.case_id/f'hour_{hour}'
                            destination.mkdir(parents=True,exist_ok=True)
                            result_path=destination/'result.json'
                            print(f'{tag} | {case.case_id} | jam {hour}',flush=True)
                            if result_path.exists():
                                result=read_json(result_path)
                                require(result['protocol_sha256']==digest(manifest)
                                        and result['case_id']==case.case_id and result['model']==tag
                                        and result['hour']==hour and result['cohort_row']==case.cohort_row
                                        and sha(destination/'attributions.npz')==result['attributions_sha256'],
                                        'Hasil XAI tersimpan berubah.')
                                records.append(result);continue
                            context=xai_context(ds,rows,local_index,cache,hour,device)
                            actual=xai_actual(context,hour);reference=xai_reference(actual,schema)
                            block=contract['settings']['block_hours']
                            with torch.no_grad():
                                expected,_=forward_prefix(model,ds,rows,cache,model_info['cfg'],device,block,hour)
                                expected=float(expected[local_index,-1])
                            with xai_gradient_backend():
                                with torch.no_grad():obtained=float(xai_forward(model,context,actual,hour,block))
                                require(math.isclose(expected,obtained,abs_tol=settings['prediction_atol'],
                                                     rel_tol=settings['prediction_rtol']),
                                        'Prediksi backend XAI berbeda dari evaluator. Analisis dihentikan.')
                                attrs,quality=xai_ig(model,context,actual,reference,hour,block,settings)
                                features=xai_feature_table(attrs,schema)
                                checks=xai_fidelity(model,context,actual,reference,schema,features,hour,block,settings) if quality['numerical_pass'] else []
                                graph=xai_graph_effect(model,context,actual,hour,block,settings) if quality['numerical_pass'] and case.clinician_case else []
                            clinical,profile=xai_clinical_context(ds,context,actual,schema,hour)
                            neighbours=[]
                            for edge_result in graph:
                                neighbour_row=int(rows[edge_result['local_neighbour']])
                                j=int(edge_result['local_neighbour'])
                                neighbour_actual={k:(context['batch'][k][j,:hour] if k in ('numeric','notes') else
                                                      context['batch'][k][j]).detach() for k in XAI_INPUT_KEYS}
                                neighbour_clinical,neighbour_profile=xai_clinical_context(ds,dict(context,local_index=j),neighbour_actual,schema,hour)
                                top_temporal=set(features[features.modality.eq('temporal')].nlargest(5,'importance').feature)
                                neighbour_profile += [dict(feature=r['feature'],value=f"{r['last_value']:.3g} (nilai terakhir; satuan sumber)")
                                                      for r in neighbour_clinical if r['feature'] in top_temporal]
                                neighbours.append(dict(neighbour_code=edge_result['neighbour_code'],
                                    profile='; '.join(r['feature']+'='+r['value'] for r in neighbour_profile)))
                            xai_save_arrays(destination/'attributions.npz',attrs)
                            features.to_csv(destination/'feature_attributions.csv',index=False)
                            pd.DataFrame(clinical).to_csv(destination/'clinical_values.csv',index=False)
                            xai_plot_local(features,destination/'local_top15.png',f'{case.case_id} | jam {hour}')
                            xai_plot_heatmap(attrs,schema['temporal_names'],destination/'temporal_heatmap.png')
                            result=dict(model=tag,case_id=case.case_id,cohort_row=int(case.cohort_row),
                                        los_group=case.los_group,hour=hour,selected_epoch=model_info['epoch'],
                                        total_los_days=float(ds.y[case.cohort_row]),quality=quality,
                                        prediction_agreement_max_abs=abs(obtained-expected),fidelity=checks,
                                        graph_effect=graph,neighbour_profiles=neighbours,clinical=clinical,profile=profile,
                                        protocol_sha256=digest(manifest),
                                        attributions_sha256=sha(destination/'attributions.npz'))
                            require({k:v._version for k,v in model.state_dict(keep_vars=True).items()}==weights_before,
                                    'Bobot model berubah selama analisis.')
                            watch_unchanged(inputs['watch']);atomic_json(result_path,result)
                            records.append(result)
                            del context,actual,reference,attrs
                    del model,saved
                    if str(device).startswith('cuda'):torch.cuda.empty_cache()
                table=xai_aggregate(report,sample,records,schema,settings,mode)
                passed=int(table.numerical_pass.sum())
                status=('PILOT_COMPLETE' if mode=='pilot' else 'XAI_COMPLETE') if passed==len(table) else 'NUMERICAL_REVIEW_REQUIRED'
                atomic_json(report/'run_metadata.json',dict(status=status,mode=mode,cases=len(active),
                            explanation_units=len(table),numerical_pass=passed,
                            protocol_sha256=digest(manifest),optimizer_updates=0,
                            primary_hour=settings['hours'][0],comparison_hour=settings['hours'][1]))
                watch_unchanged(inputs['watch'])
                print(f'{status}: {passed}/{len(table)} penjelasan lolos pemeriksaan numerik.',flush=True)
                return report,table
            finally:ds.close()

## Jalankan analisis

In [76]:
REPORT, results = run_step07(ROOT, BASELINE_RUN, DGT128_RUN, MODE, XAI_SETTINGS)
print("Laporan:", REPORT)
display(results[["model", "case_id", "hour", "selected_epoch", "prediction_days",
                 "completeness_error_days", "numerical_pass"]].round(5))
print("Buka paket dokter:", REPORT / "doctor_packets" / "index.html")

Input integrity 1/851 files
Input integrity 51/851 files
Input integrity 101/851 files
Input integrity 151/851 files
Input integrity 201/851 files
Input integrity 251/851 files
Input integrity 301/851 files
Input integrity 351/851 files
Input integrity 401/851 files
Input integrity 451/851 files
Input integrity 501/851 files
Input integrity 551/851 files
Input integrity 601/851 files
Input integrity 651/851 files
Input integrity 701/851 files
Input integrity 751/851 files
Input integrity 801/851 files
Input integrity 851/851 files
Mode full: 90 kasus x dua model x dua landmark. Laporan: C:\Users\ASUS\Documents\icu_los\outputs\07_xai_70_15_15\c6943951feac7a75
DGT64 | K001 | jam 48
DGT64 | K001 | jam 24
DGT64 | K002 | jam 48
    IG 32: error=4.32246e-07 hari; stabilitas=None
    IG 64: error=3.87333e-07 hari; stabilitas=5.7346723462534945e-08
DGT64 | K002 | jam 24
    IG 32: error=4.09181e-08 hari; stabilitas=None
    IG 64: error=1.31512e-08 hari; stabilitas=1.0921205830847984e-07
DGT64

,model,case_id,hour,selected_epoch,prediction_days,completeness_error_days,numerical_pass
0,DGT64,K001,48,9,4.22858,0.00000,True
1,DGT64,K001,24,9,4.29696,0.00000,True
2,DGT64,K002,48,9,2.97568,0.00000,True
3,DGT64,K002,24,9,1.88412,0.00000,True
4,DGT64,K003,48,9,2.92011,0.00000,True
...,...,...,...,...,...,...,...
355,DGT128,K088,24,9,2.96382,0.00000,True
356,DGT128,K089,48,9,12.52922,0.00051,True
357,DGT128,K089,24,9,8.17910,0.00000,True
358,DGT128,K090,48,9,7.39635,0.00000,True


Buka paket dokter: C:\Users\ASUS\Documents\icu_los\outputs\07_xai_70_15_15\c6943951feac7a75\doctor_packets\index.html
